# Optimized Zero-Encoding for Polynomial Masking (CHES Artifact)

Copyright 2025-2026 CAC – Chair of Applied Cryptography

SPDX-License-Identifier: MIT

This Jupyter notebook is the **executable research artifact** accompanying the CHES/TCHES paper
**“UP TO 50% OFF: Efficient Implementation of Polynomial Masking.”**

All implementations are aligned with the notation, algorithms, and
security arguments presented in the paper, in particular:

- Zero-Encodings (ZEnc), Optimized Zero-Encodings (opt-ZEnc)
- Strong Zero-Encodings (sZEnc, opt-sZEnc)
- Packed Secret Sharing (PSS)
- LaOla / ISW / BGW-style multiplication and degree reduction
- Combined (t, e)-adversary model

It provides SageMath implementations of the core constructions introduced in the paper, with a particular focus on the **optimized zero-encoding (opt-ZEnc)** scheme described in **Section 3.1**.
The notebook includes both the algorithmic implementations and automated correctness checks.

The artifact is **fully self-contained** and can be executed top-to-bottom to reproduce the functional correctness tests and experimental validations reported in the paper.

---

## Notation and Conventions

Throughout this notebook, we follow the notation and parameter conventions used in the paper:

1. **Number of shares:** `n` (`num_shares`)
2. **Number of secrets:** `k` (`num_secrets`)
3. **Polynomial degree:** `d` (`degree`)
4. **Secret support points:** `secrets_supports`
   (evaluation points at which the secrets are embedded)
5. **Share support points:** `shares_supports`
   (evaluation points at which the shares are generated)
6. **Share vector:** `shares_n`
   (the list of `n` evaluated shares)
7. **Number of faults:** `num_faults`

Derived parameters follow the paper’s conventions, in particular
$$
t = d + 1 - k \quad \text{and} \quad n = d + e + 1 .
$$

All arithmetic is performed over a finite field, instantiated in this notebook using SageMath’s `FiniteField`.

This jupyter notebook contains all the algorithms in sagemath for paper - UP TO 50% OFF: Efficient Implementation of Polynomial Masking <be>
Notation:
1. number of shares: num_shares, n
2. number of secrets: num_secrets, k
3. polynomial degree: degree, d
4. support points embedding the secrets: secrets_supports
5. support points generating the shares: shares_supports
6. n shares: shares_n
7. number of faults: num_faults

## Import packages

The following cell imports all external dependencies required by this artifact.

In [1]:
# ============================================================
# Imports
# ============================================================

# SageMath finite field (GF(2^m)) utilities
from sage.rings.finite_rings.all import FiniteField
from sage.rings.finite_rings.element_givaro import FiniteField_givaroElement as FiniteFieldElement

# Numerical utilities
import numpy as np

# Randomness and combinatorics (testing utilities)
import random
from collections import Counter
from itertools import combinations

# Progress display
from tqdm import tqdm

## Primitive Operations over Finite Fields and Shares

In [2]:
# ------------------------------------------------------------
# Field element conversion utilities
# ------------------------------------------------------------

def int_to_poly(ff, i: int) -> FiniteFieldElement:
    """
    Transforms a non-negative integer into its polynomial representation
    as an element of the finite field ff.

    The integer is interpreted via its binary expansion in little-endian
    bit order and mapped to a polynomial over GF(2).

    :param i: the non-negative integer to be transformed
    :return: the finite-field element representing the polynomial encoding of i
    """
    return ff([int(i) for i in list(bin(i))[:1:-1]])


def int_to_polylist(ff, i: int, list_length) -> FiniteFieldElement:
    """
    Replicates the polynomial representation of an integer i into a list
    of given length.

    :param i: the integer that has to be transformed
    :return: the list of identical polynomial representations of i
    """
    return [int_to_poly(ff, i) for j in range(list_length)]


def intlist_to_polylist(ff, intlist: list) -> list:
    """
    Converts a list of integers into their corresponding polynomial
    representations over the finite field ff.

    :param list: the list of the integers that has to be transformed
    :return: the list of polynomial representations of the input integers
    """
    return [int_to_poly(ff, I) for I in intlist]


def polynomial_multiplication(ff, a, b):
    """
    Multiplies two integers a and b in the finite field ff and returns
    the result as an integer.

    :param ff: finite field
    :return: the integer representation of the field product a · b
    """
    a_poly = int_to_poly(ff, a)
    b_poly = int_to_poly(ff, b)
    return (a_poly * b_poly).to_integer()


def polynomial_addition(ff, a, b):
    """
    Adds two integers a and b in finite field arithmetic and returns
    the result as an integer.

    :param ff: finite field
    :return: the integer representation of the field sum a + b
    """
    a_poly = int_to_poly(ff, a)
    b_poly = int_to_poly(ff, b)
    return (a_poly + b_poly).to_integer()

In [3]:
# ------------------------------------------------------------
# Low-level field operations (semantic wrappers)
# ------------------------------------------------------------

def poly_add(a, b):
    """
    Field addition in GF(2^m).

    Parameters
    ----------
    a, b : Finite field elements
        Elements in the same finite field.

    Returns
    -------
    Finite field element
        a + b
    """
    return a + b


def poly_mult(a, b):
    """
    Field multiplication in GF(2^m).

    Parameters
    ----------
    a, b : Finite field elements

    Returns
    -------
    Finite field element
        a * b
    """
    return a * b


def poly_power(a, e: int):
    """
    Exponentiation in GF(2^m).

    Parameters
    ----------
    a : Finite field element
    e : int
        Non-negative integer exponent.

    Returns
    -------
    Finite field element
        a^e
    """
    if not isinstance(e, (int, Integer)) or e < 0:
        raise ValueError(f"Exponent e must be a non-negative integer, got {e!r}")
    return a ** e


def poly_square(a):
    """
    Frobenius squaring in GF(2^m): a -> a^2.

    Parameters
    ----------
    a : Finite field element

    Returns
    -------
    Finite field element
        a^2
    """
    return a * a   # or: return a**2


In [4]:
# ------------------------------------------------------------
# Share-wise operations
# ------------------------------------------------------------

def sw_add(a, b):
    """
    Share-wise addition of two share vectors.

    :param a: list of field elements (shares)
    :param b: list of field elements (shares)
    :return: share-wise sum a + b
    """
    assert len(a) == len(b), "Share vectors must have the same length"
    return [a[i] + b[i] for i in range(len(a))]


def sw_mult(a, b):
    """
    Share-wise multiplication of two share vectors.

    :param a: list of field elements (shares)
    :param b: list of field elements (shares)
    :return: share-wise product a · b
    """
    assert len(a) == len(b), "Share vectors must have the same length"
    return [a[i] * b[i] for i in range(len(a))]


def unary_add(a, c):
    """
    Adds a vector of field elements to a share vector.

    :param a: list of field elements (shares)
    :param c: list of field elements (same length as a)
    :return: share-wise sum a + c
    """
    assert len(a) == len(c), "Operands must have the same length"
    return [a[i] + c[i] for i in range(len(a))]


def unary_mult(a, c):
    """
    Multiplies a share vector with a vector of field elements.

    :param a: list of field elements (shares)
    :param c: list of field elements (same length as a)
    :return: share-wise product a · c
    """
    assert len(a) == len(c), "Operands must have the same length"
    return [a[i] * c[i] for i in range(len(a))]

In [5]:
# ============================================================
# Parameter Validation Utilities
# ============================================================
#
# This cell defines helper functions for validating the parameter
# tuple (n, d, k) used throughout the notebook:
#
#   n : number of shares
#   d : polynomial degree (indegree)
#   k : number of embedded secrets
#
# The checks implemented here enforce the structural assumptions
# required by Shamir sharing, optimized zero-encoding (optZEnc),
# and the SplitRed gadget used in this artifact.
#
# Centralizing these constraints ensures:
#   - consistent parameter validation across all tests,
#   - clearer error messages when assumptions are violated,
#   - easier maintenance when constraints evolve.
#
# The `strict` flag allows selectively disabling SplitRed-specific
# constraints when testing partial components in isolation.
# ============================================================

def is_valid_params(num_shares, degree, num_secrets, *, strict=True):
    """
    Check whether (num_shares, degree, num_secrets) satisfies the constraints
    required by the gadgets/tests in this notebook.

    Parameters
    ----------
    num_shares : int  # n
    degree     : int  # d (indegree)
    num_secrets: int  # k
    strict : bool
        If True, enforce the SplitRed requirement k <= floor(d/2).
        If False, skip that constraint (useful if testing partial components only).

    Returns
    -------
    (ok, reasons)
      ok: bool
      reasons: list[str]  (empty if ok)
    """
    n = int(num_shares)
    d = int(degree)
    k = int(num_secrets)

    reasons = []

    # Basic sanity
    if n <= 0 or d <= 0 or k <= 0:
        reasons.append("n, d, k must all be positive integers.")

    # Need enough shares to represent degree-d Shamir polynomial: n >= d + 2
    # (your original filter used n - d - 1 > 0)
    if n - d - 1 <= 0:
        reasons.append("Require n - d - 1 > 0 (i.e., n >= d + 2) to hold degree-d sharing.")

    # Need degree large enough to carry k secrets in your encoding convention
    # (your original filter used d - k + 1 > 1, i.e., d >= k + 1)
    if d - k + 1 <= 1:
        reasons.append("Require d - k + 1 > 1 (i.e., d >= k + 1) so degree d can hide k secrets.")

    # SplitRed requirement: after reduction to floor(d/2), still enough room for k secrets
    if strict and k > (d // 2):
        reasons.append("Require k <= floor(d/2) for SplitRed degree reduction to still hide k secrets.")

    return (len(reasons) == 0), reasons


def assert_valid_params(num_shares, degree, num_secrets, *, strict=True):
    """
    Raise ValueError with a readable message if parameters are invalid.
    """
    ok, reasons = is_valid_params(num_shares, degree, num_secrets, strict=strict)
    if not ok:
        msg = (
            f"Invalid parameters (n={num_shares}, d={degree}, k={num_secrets}).\n"
            + "\n".join(f"  - {r}" for r in reasons)
        )
        raise ValueError(msg)


In [6]:
# ------------------------------------------------------------
# Matrix and data-structure utilities
# ------------------------------------------------------------

def print_polynomial_matrix(label, poly_matrix):
    """
    Pretty-print a Sage matrix (or vector) over a finite field by converting each entry
    to its integer encoding via .to_integer() when available.

    This is mainly a debugging helper: it prints the matrix in integer form.

    :param label: label printed before the matrix
    :param poly_matrix: Sage matrix or vector over a finite field
    :return: None
    """
    print(label)

    # Accept either a matrix or a vector-like object.
    if hasattr(poly_matrix, "nrows") and hasattr(poly_matrix, "ncols"):
        nrows, ncols = poly_matrix.nrows(), poly_matrix.ncols()
        integer_matrix = zero_matrix(ZZ, nrows, ncols)
        for i in range(nrows):
            for j in range(ncols):
                x = poly_matrix[i, j]
                integer_matrix[i, j] = x.to_integer() if hasattr(x, "to_integer") else ZZ(x)
        print(integer_matrix)
        return

    # Fallback: treat as iterable (e.g., vector)
    vec = list(poly_matrix)
    integer_vec = [x.to_integer() if hasattr(x, "to_integer") else int(x) for x in vec]
    print(integer_vec)


def copy_matrix(ff, A_tilde):
    """
    Deep copy a Sage matrix over the finite field ff.

    Prefer the matrix's native copy when available (faster and clearer),
    otherwise fall back to an explicit element-wise copy.

    :param ff: finite field
    :param A_tilde: Sage matrix over ff
    :return: deep copy of A_tilde
    """
    if hasattr(A_tilde, "copy"):
        return A_tilde.copy()

    nrows, ncols = A_tilde.nrows(), A_tilde.ncols()
    A_copy = zero_matrix(ff, nrows, ncols)
    for i in range(nrows):
        for j in range(ncols):
            A_copy[i, j] = A_tilde[i, j]
    return A_copy


def create_list_list(n1, n2, fill=0):
    """
    Create a nested Python list with shape (n2 x n1).

    :param n1: number of columns
    :param n2: number of rows
    :param fill: fill value (default: 0)
    :return: list of lists (each row is independent)
    """
    return [[fill for _ in range(n1)] for _ in range(n2)]

In [7]:
# ------------------------------------------------------------
# Random sampling helpers (used for tests / experiments)
# ------------------------------------------------------------

def _as_1d_candidates(support_point_range):
    """
    Normalize support_point_range to a 1-D numpy array of candidates.
    Accepts lists, ranges, numpy arrays, or general iterables.
    """
    arr = np.array(list(support_point_range), dtype=int)
    if arr.ndim != 1:
        raise ValueError("support_point_range must be 1-D (or an iterable of scalars).")
    if arr.size == 0:
        raise ValueError("support_point_range must be non-empty.")
    return arr


def generate_random_field_elements(ff, support_point_range, n):
    """
    Sample n distinct support points uniformly at random and convert them
    into field elements over ff.

    NOTE: For testing/experiments only (not cryptographic randomness).
    """
    candidates = _as_1d_candidates(support_point_range)

    if n < 0:
        raise ValueError("n must be non-negative.")
    if n > candidates.size:
        raise ValueError(f"Cannot sample n={n} distinct points from only {candidates.size} candidates.")

    random_ints = np.random.choice(candidates, size=n, replace=False)
    return intlist_to_polylist(ff, random_ints.tolist())


def generate_random_field_elements_listlist(ff, support_point_range, n1, n2):
    """
    Generate an n2-by-n1 nested list of random field elements, each row sampled
    without replacement from the same candidate pool.

    NOTE: For testing/experiments only (not cryptographic randomness).
    """
    candidates = _as_1d_candidates(support_point_range)

    if n1 < 0 or n2 < 0:
        raise ValueError("n1 and n2 must be non-negative.")
    if n1 > candidates.size:
        raise ValueError(f"Cannot sample n1={n1} distinct points from only {candidates.size} candidates.")

    rows = []
    for _ in range(n2):
        random_ints = np.random.choice(candidates, size=n1, replace=False)
        rows.append(intlist_to_polylist(ff, random_ints.tolist()))
    return rows

In [8]:
# ------------------------------------------------------------
# Combinatorics helpers (partial shares and indexing)
# ------------------------------------------------------------

def create_shares_supports_list_for_partial_shares(
    shares_n, shares_supports, num_shares, num_shares_partial
):
    """
    Enumerate all subsets of size `num_shares_partial` from `num_shares` shares.

    Returns two parallel lists:
      - list_partial_shares:     each entry is a sublist of shares
      - list_partial_supports:   the corresponding support points

    :param shares_n: list of n shares
    :param shares_supports: list of n share support points
    :param num_shares: n (number of shares)
    :param num_shares_partial: subset size
    :return: (list_partial_shares, list_partial_supports)
    """
    if len(shares_n) != num_shares:
        raise ValueError("shares_n length must match num_shares")
    if len(shares_supports) != num_shares:
        raise ValueError("shares_supports length must match num_shares")
    if not (0 <= num_shares_partial <= num_shares):
        raise ValueError("invalid subset size: must satisfy 0 <= num_shares_partial <= num_shares")

    list_partial_shares = []
    list_partial_supports = []

    for idx_tuple in combinations(range(num_shares), num_shares_partial):
        list_partial_shares.append([shares_n[i] for i in idx_tuple])
        list_partial_supports.append([shares_supports[i] for i in idx_tuple])

    return list_partial_shares, list_partial_supports


def sort_list_by_reference(ref_list, target_list):
    """
    Return indices of elements in `target_list` following the order in `ref_list`.

    Example:
      ref_list    = [b, a]
      target_list = [a, b, c]
      -> [1, 0]

    :param ref_list: reference ordering (elements must appear in target_list)
    :param target_list: list to be indexed
    :return: list of indices in target_list
    :raises ValueError: if some element of ref_list is missing in target_list
    """
    index_of = {x: i for i, x in enumerate(target_list)}
    try:
        return [index_of[x] for x in ref_list]
    except KeyError as e:
        raise ValueError(f"Element {e.args[0]!r} from ref_list is not in target_list") from None

## AES S-box (unmasked reference)

This section provides an unmasked AES S-box reference implementation over GF(2^8),
used only for validation and demonstration (not required by opt-ZEnc itself).

In [9]:
# ------------------------------------------------------------
# AES S-box (unmasked reference implementation)
# ------------------------------------------------------------

def sbox(ff, f):
    """
    Unmasked AES S-box over GF(2^8).

    Computes the AES multiplicative inverse f^(254) (with 0 -> 0),
    followed by the AES affine transform, expressed as a linearized
    polynomial over GF(2^8).

    :param ff: Sage finite field GF(2^8)
    :param f:  field element in ff
    :return: S-box output in ff
    """
    # --- inversion: f^(254) ---
    f2   = poly_square(f)                # f^2
    f4   = poly_square(f2)               # f^4
    f8   = poly_square(f4)               # f^8
    f9   = poly_mult(f8, f)              # f^9
    f18  = poly_square(f9)               # f^18
    f19  = poly_mult(f18, f)             # f^19
    f36  = poly_square(f18)              # f^36
    f55  = poly_mult(f36, f19)           # f^55
    f72  = poly_square(f36)              # f^72
    f127 = poly_mult(f72, f55)           # f^127
    inv  = poly_square(f127)             # f^254

    # --- affine transform (linearized polynomial form) ---
    res = ff(0)
    res = poly_add(res, int_to_poly(ff, 99))

    tmp = poly_mult(inv, int_to_poly(ff, 5))
    res = poly_add(res, tmp)

    inv2 = poly_square(inv)
    tmp  = poly_mult(inv2, int_to_poly(ff, 9))
    res  = poly_add(res, tmp)

    inv4 = poly_square(inv2)
    tmp  = poly_mult(inv4, int_to_poly(ff, 249))
    res  = poly_add(res, tmp)

    inv8 = poly_square(inv4)
    tmp  = poly_mult(inv8, int_to_poly(ff, 37))
    res  = poly_add(res, tmp)

    inv16 = poly_square(inv8)
    tmp   = poly_mult(inv16, int_to_poly(ff, 244))
    res   = poly_add(res, tmp)

    inv32 = poly_square(inv16)
    tmp   = poly_mult(inv32, int_to_poly(ff, 1))
    res   = poly_add(res, tmp)

    inv64 = poly_square(inv32)
    tmp   = poly_mult(inv64, int_to_poly(ff, 181))
    res   = poly_add(res, tmp)

    inv128 = poly_square(inv64)
    tmp    = poly_mult(inv128, int_to_poly(ff, 143))
    res    = poly_add(res, tmp)

    return res

## Precomputed Linear Maps (Vandermonde, Encoding/Decoding, Zero-Encoding, Degree Reduction)

This section defines constant matrices and linear maps that depend **only** on public support points and parameters ((n,k,d)).
They can be precomputed once per parameter set and reused across experiments.

In [10]:
# ------------------------------------------------------------
# Vandermonde-based helpers and Frobenius permutations
#
# This cell defines low-level linear-algebra utilities that depend
# only on public support points. They are used to construct
# encoding/decoding maps, zero-encodings, degree-reduction matrices,
# and to implement Frobenius squaring as a permutation.
#
# All functions here are deterministic and parameter-only; they can
# be precomputed once per (n, k, d, supports) setting.
# ------------------------------------------------------------

def generate_permutation_map(supports):
    """
    Return the permutation map induced by the Frobenius squaring map over GF(2^m).

    For a list of support points `supports = [x_0, ..., x_{n-1}]`, this function
    returns a list `perm` of length n such that:
        supports[perm[i]] == supports[i]**2

    This is useful for implementing squaring as a pure index permutation when
    supports are closed under squaring.

    :param supports: list of field elements (support points)
    :return: list of indices perm (same length as supports)
    :raises ValueError: if supports are not closed under squaring
    """
    # Precompute element -> index to avoid repeated O(n) lookups.
    index_of = {x: idx for idx, x in enumerate(supports)}

    perm = []
    for x in supports:
        x2 = x**2
        if x2 not in index_of:
            raise ValueError("supports are not closed under squaring: missing x^2 in supports.")
        perm.append(index_of[x2])
    return perm


def generate_vandermonde(ff, supports, n, m):
    """
    Return the n-by-m Vandermonde matrix over `ff` built from `supports`.

    The (i, j)-entry is supports[i]^j for i=0..n-1 and j=0..m-1.

    Note: Sage's `matrix.vandermonde(v, ff)` returns a square matrix of size len(v).
    We therefore build a square Vandermonde of size L=max(n,m) using enough points
    and then take the top-left n-by-m submatrix.

    :param ff: finite field
    :param supports: list of candidate support points (field elements)
    :param n: number of rows
    :param m: number of columns
    :return: Sage matrix over ff with shape (n x m)
    :raises ValueError: if supports has fewer than n elements
    """
    if n < 0 or m < 0:
        raise ValueError("n and m must be non-negative.")
    if len(supports) < n:
        raise ValueError("supports must contain at least n elements.")

    L = max(n, m)
    # Ensure the Vandermonde constructor receives exactly L points.
    v_points = supports[:n] + [ff(0)] * (L - n)
    v_square = matrix.vandermonde(v_points, ff)  # L x L
    return v_square.submatrix(0, 0, n, m)


def generate_Vs(ff, shares_supports, num_shares):
    """
    Construct the Vandermonde matrix of share supports and its inverse.

    :param ff: finite field
    :param shares_supports: list of n share support points
    :param num_shares: n
    :return: (V, V_inv) where V is n-by-n and V_inv = V^{-1}
    """
    V = generate_vandermonde(ff, shares_supports, num_shares, num_shares)
    return V, V.inverse()

In [11]:
# ------------------------------------------------------------
# Encoding / Decoding linear maps (masking/unmasking)
#
# These depend only on public support points and (n,k,d).
# You can precompute them once per parameter set.
# ------------------------------------------------------------

def generate_M_enc(ff, secrets_supports, shares_supports, num_secrets, num_shares, degree):
    """
    Construct the masking/encoding matrix M_enc (A') for masking/encoding (cf. Section 2.2, Section 3.1)

    The construction follows the usual "interpolation then evaluation" pattern:
      - U maps coefficients -> values at (secret supports + some share supports)
      - V maps coefficients -> values at remaining share supports
      - M_enc = V * U^{-1}

    :param ff: finite field
    :param secrets_supports: support points for embedding secrets
    :param shares_supports: support points for generating shares
    :param num_secrets: k
    :param num_shares: n
    :param degree: d (polynomial degree)
    :return: M_enc as a Sage matrix over ff
    """
    if not (0 <= num_secrets <= degree + 1):
        raise ValueError("num_secrets must satisfy 0 <= k <= d+1.")
    if num_shares <= 0:
        raise ValueError("num_shares must be positive.")

    # U is (d+1) x (d+1)
    U_points = secrets_supports[:num_secrets] + shares_supports[: (degree + 1 - num_secrets)]
    U = generate_vandermonde(ff, U_points, degree + 1, degree + 1)
    U_inv = U.inverse()

    # V is (n - (d+1-k)) x (d+1)
    offset = degree + 1 - num_secrets
    V_points = shares_supports[offset:]
    V = generate_vandermonde(ff, V_points, num_shares - offset, degree + 1)

    return V * U_inv

def generate_M_dec(ff, secrets_supports, shares_supports, num_shares, num_secrets):
    """
    Construct the decoding/unmasking matrix M_dec.

    Given share values y (length n):
      - V^{-1} recovers polynomial coefficients from shares
      - U selects/evaluates the secret coordinates from the coefficients
      - M_dec = U * V^{-1}

    :param ff: finite field
    :param secrets_supports: support points for secrets
    :param shares_supports: support points for shares
    :param num_shares: n
    :param num_secrets: k
    :return: M_dec as a Sage matrix over ff
    """
    V = generate_vandermonde(ff, shares_supports, num_shares, num_shares)
    V_inv = V.inverse()

    U = generate_vandermonde(ff, secrets_supports[:num_secrets], num_secrets, num_shares)
    return U * V_inv

def generate_A_tilde(ff, M_enc, num_secrets, degree):
    """
    Construct the matrix A_tilde used for zero-encoding
    (cf. Section 3.1, Algorithm 1).

    A_tilde depends on the zero-encoding parameters and must be regenerated
    whenever (num_shares, num_secrets, degree) change. It is formed by stacking:
      - an identity block of size (d+1-k), corresponding to high-degree randomness
      - the submatrix of M_enc obtained by removing the first k columns

    :param ff: finite field
    :param M_enc: encoding matrix returned by generate_M_enc
    :param num_secrets: k (number of secrets)
    :param degree: d (polynomial degree)
    :return: A_tilde as a Sage matrix over ff
    """
    if not (0 <= num_secrets <= degree + 1):
        raise ValueError("num_secrets must satisfy 0 <= k <= d+1.")

    # Drop the first k columns: shape (?) x (d+1-k)
    mat_d = M_enc.submatrix(0, num_secrets)

    # Identity block of size (d+1-k)
    r = degree + 1 - num_secrets
    mat_u = identity_matrix(ff, r)

    return block_matrix([[mat_u], [mat_d]], subdivide=False)

In [12]:
# ============================================================
# Degree reduction + error propagation linear maps
# (Packed Secret Sharing / Shamir; cf. ePrint 2025/035 + OhLaLa)
# ============================================================

def generate_M_lambda(ff, secrets_supports, shares_supports, num_shares, indegree, outdegree):
    """
    Compute the linear map M_lambda that reduces the polynomial degree while preserving the embedded secrets.

    Given n shares (evaluations at shares_supports) of a polynomial f of degree <= indegree (e.g., d),
    this function returns a matrix M_lambda such that:

        M_lambda * shares = coeffs(g)

    where g is a polynomial of degree <= outdegree (e.g., floor(d/2)) that embeds the same secrets
    at secrets_supports as f. Intuitively, the construction (i) reconstructs the low-degree coefficient vector
    from shares via the inverse Vandermonde of the share supports, (ii) converts coefficients <-> values on the
    secret support points via Vandermonde matrices, and (iii) truncates to degree outdegree by zero-padding.

    Reference: Appendix A.2 of “All-You-Can-Compute: Packed Secret Sharing for Combined Resilience”
    (ePrint 2025/035).

    :param ff: finite field
    :param secrets_supports: support points for secrets (length >= indegree+1)
    :param shares_supports: support points for shares (length = num_shares)
    :param num_shares: n (number of shares)
    :param indegree: input polynomial degree bound (e.g., d)
    :param outdegree: target reduced degree bound (e.g., floor(d/2))
    :return: M_lambda as a Sage matrix over ff (shape (outdegree+1) x num_shares)
    """
    # Step 1 (unmask / interpolate): invert the Vandermonde on share supports to map shares -> coefficients.
    # mat_V_inv * shares = full coefficient vector (with implicit high-degree zeros if indegree < n-1).
    mat_V_inv = generate_vandermonde(ff, shares_supports, num_shares, num_shares).inverse()

    # Step 2 (restrict to available degree): keep only rows for coefficients up to indegree.
    # mat_M1 * shares = coefficients of degree <= indegree.
    mat_M1 = mat_V_inv.submatrix(0, 0, indegree + 1, -1)

    # Step 3 (switch basis via secret support points):
    #   - U_indegree maps coefficients -> values on secrets_supports (size indegree+1)
    #   - U_outdegree^{-1} maps values -> coefficients of degree <= outdegree (size outdegree+1)
    mat_U_indegree_x_indegree = generate_vandermonde(ff, secrets_supports, indegree + 1, indegree + 1)
    mat_U_outdegree_x_outdegree = generate_vandermonde(ff, secrets_supports, outdegree + 1, outdegree + 1)
    mat_U_outdegree_x_outdegree_inv = mat_U_outdegree_x_outdegree.inverse()

    # Step 4 (degree reduction): embed the outdegree inverse transform into an (indegree+1)-dim space
    # by padding with zeros, so it can multiply the indegree-domain vector.
    zero_pad_right = matrix(ff, outdegree + 1, indegree - outdegree, 0)
    zero_pad_bottom = matrix(ff, indegree - outdegree, indegree + 1, 0)
    mat_M2 = block_matrix(
        [[mat_U_outdegree_x_outdegree_inv, zero_pad_right],
         [zero_pad_bottom]],
        subdivide=False
    )

    # Step 5 (compose): shares -> coeffs(f) -> values(secrets_supports) -> coeffs(g) (degree <= outdegree).
    mat_M3 = mat_M2 * (mat_U_indegree_x_indegree * mat_M1)

    # Output: keep only the outdegree+1 coefficient rows.
    return mat_M3.submatrix(0, 0, outdegree + 1, -1)


def generate_M_lambda_with_permutation(
    ff,
    secrets_supports_in,
    secrets_supports_out,
    shares_supports,
    num_shares,
    indegree,
    outdegree,
):
    """
    Construct the linear map M_lambda that turns an n-share vector (evaluations of a degree-≤indegree polynomial
    embedding secrets at secrets_supports_in) into the coefficients of a degree-≤outdegree polynomial embedding
    the *same* secrets at secrets_supports_out.

    Intuition (matches your comments):
      - interpolate from shares_supports to recover coefficients up to degree indegree;
      - evaluate/re-encode the secret part using secrets_supports_out and truncate to degree outdegree;
      - pack the two steps into one matrix multiplication.

    Parameters are kept explicit to mirror the paper + keep API stable.

    :return: M_lambda of shape (outdegree+1) x num_shares
    """
    # --- sanity checks (fail early, nicer debugging) ---
    if outdegree > indegree:
        raise ValueError(f"outdegree must be <= indegree, got outdegree={outdegree}, indegree={indegree}")
    if num_shares <= indegree:
        raise ValueError(f"need num_shares > indegree to interpolate, got num_shares={num_shares}, indegree={indegree}")
    if len(shares_supports) < num_shares:
        raise ValueError("shares_supports shorter than num_shares")
    if len(secrets_supports_in) < indegree + 1:
        raise ValueError("secrets_supports_in must provide at least indegree+1 points")
    if len(secrets_supports_out) < outdegree + 1:
        raise ValueError("secrets_supports_out must provide at least outdegree+1 points")

    # --- Step 1: interpolation from shares to coefficients (truncate to degrees 0..indegree) ---
    # V^{-1} maps shares -> full coefficient vector (length n); keep only rows for degrees 0..indegree.
    V_inv = generate_vandermonde(ff, shares_supports, num_shares, num_shares).inverse()
    shares_to_coeffs = V_inv.submatrix(0, 0, indegree + 1, -1)  # (indegree+1) x n

    # --- Step 2: change the encoding supports + truncate to outdegree ---
    # Convert coeffs w.r.t. secrets_supports_in (degree indegree) into evaluations at secrets_supports_out (degree outdegree),
    # via: coeffs_in -> values_on_in -> coeffs_out(truncated)
    U_in  = generate_vandermonde(ff, secrets_supports_in,  indegree + 1, indegree + 1)   # (indegree+1) x (indegree+1)
    U_out = generate_vandermonde(ff, secrets_supports_out, outdegree + 1, outdegree + 1) # (outdegree+1) x (outdegree+1)
    U_out_inv = U_out.inverse()

    # Embed U_out^{-1} into an (indegree+1) x (indegree+1) matrix by padding zeros (acts only on low degrees).
    pad_right  = matrix(ff, outdegree + 1, indegree - outdegree, 0)
    pad_bottom = matrix(ff, indegree - outdegree, indegree + 1, 0)
    truncate_to_outdegree = block_matrix([[U_out_inv, pad_right], [pad_bottom]], subdivide=False)

    # --- Compose: shares -> coeffs_in -> coeffs_out(truncated) ---
    # M_lambda = (truncate_to_outdegree * U_in) * shares_to_coeffs
    M_lambda_full = truncate_to_outdegree * (U_in * shares_to_coeffs)  # (indegree+1) x n
    M_lambda = M_lambda_full.submatrix(0, 0, outdegree + 1, -1)        # (outdegree+1) x n

    return M_lambda

In [13]:
# ============================================================================
# Linear Maps for Degree Reduction and Fault Propagation (λ, E, λ̂)
# ============================================================================

def generate_error_propagation_matrix(indegree, V, V_inv):
    """
    Construct the error-propagation matrix E for CompE (Appendix A.1) in
    “All-You-Can-Compute: Packed Secret Sharing for Combined Resilience”.

    E captures how faults on the share vector influence the high-degree part of the
    reconstructed polynomial when interpolating from n shares.

    :param indegree: input degree bound d (CompE operates on degrees > d)
    :param V: Vandermonde matrix on shares_supports (n x n)
    :param V_inv: inverse of V
    :return: E = V[:, d+1:] * V_inv[d+1:, :]  (shape n x n)
    """
    # Select the high-degree columns of V corresponding to degrees (d+1 .. n-1).
    V_hi = V.submatrix(0, indegree + 1, -1, -1)

    # Select the matching rows of V^{-1} that map shares -> high-degree coefficients.
    V_inv_hi = V_inv.submatrix(indegree + 1, 0, -1, -1)

    # Compose to obtain the error-propagation operator on the share vector.
    return V_hi * V_inv_hi


def generate_lambda_hat_matrix(V, M_lambda, error_propagation_matrix, outdegree):
    """
    Compute the λ̂ (“lambda-hat”) linear map from Section 4.2 of
    “All-You-Can-Compute: Packed Secret Sharing for Combined Resilience” (ePrint 2025/035).

    λ̂ combines:
      (1) degree reduction (via M_lambda, mapping shares -> coefficients of the reduced-degree polynomial), and
      (2) error propagation (via the precomputed error_propagation_matrix).

    Concretely, we first map reduced-degree coefficients back to an n-share vector using
    the truncated Vandermonde matrix V[:, 0..outdegree], then add the error-propagation term:

        λ̂ = V[:, :outdegree+1] · M_lambda  +  E

    :param V: Vandermonde matrix on shares_supports (n x n)
    :param M_lambda: degree-reduction map producing reduced-degree coefficients
                    (shape (outdegree+1) x n, so V_crop * M_lambda is n x n)
    :param error_propagation_matrix: E term (n x n)
    :param outdegree: target reduced degree bound
    :return: lambda_hat_matrix (n x n)
    """
    # Truncated Vandermonde: map coeffs(deg <= outdegree) -> n evaluations (shares).
    V_crop = V.submatrix(0, 0, -1, outdegree + 1)  # shape: n x (outdegree+1)

    # Degree-reduction contribution in share domain.
    M_F_to_F_tilde = V_crop * M_lambda             # shape: n x n

    # Add the error-propagation contribution.
    return M_F_to_F_tilde + error_propagation_matrix

# ------------------------------------------------------------
# Shamir degree reduction (single-shot lambda matrix)
# ------------------------------------------------------------
def generate_lambda_degree_reduction_shamir(ff, V, V_inv, num_shares, indegree, outdegree, debug=False):
    """
    Build the Shamir degree-reduction matrix λ^{d1→d2}.

    Given n shares evaluated at public support points, with underlying polynomial degree d1 (=indegree),
    λ^{d1→d2} maps the share vector to a new share vector that corresponds to the *same secrets* but with
    polynomial degree d2 (=outdegree). For Shamir, secret and share support points coincide, so this matrix
    depends only on public supports and can be precomputed.

    Conceptually:
      1) interpolate coefficients from shares (via V^{-1})
      2) truncate to degree d2 (keep first d2+1 coefficients)
      3) re-evaluate to shares (via cropped V)

    :param ff: finite field
    :param V: Vandermonde matrix on share supports (n x n)
    :param V_inv: inverse Vandermonde matrix (n x n)
    :param num_shares: n
    :param indegree: d1 (input degree)
    :param outdegree: d2 (output degree), must satisfy 0 <= d2 <= d1 < n
    :param debug: if True, print cropped matrices
    :return: lambda_degree_reduction (n x n) matrix over ff
    """
    if not (0 <= outdegree <= indegree):
        raise ValueError("Require 0 <= outdegree <= indegree.")
    if not (indegree < num_shares):
        raise ValueError("Require indegree < num_shares for Shamir interpolation.")
    if V.nrows() != num_shares or V.ncols() != num_shares:
        raise ValueError("V must be (num_shares x num_shares).")
    if V_inv.nrows() != num_shares or V_inv.ncols() != num_shares:
        raise ValueError("V_inv must be (num_shares x num_shares).")

    # Keep only coefficient degrees 0..d2 (first d2+1 columns/rows).
    V_crop = V[:, : outdegree + 1]            # (n x (d2+1))
    V_inv_crop = V_inv[: outdegree + 1, :]    # ((d2+1) x n)

    if debug:
        print_polynomial_matrix("V_crop", matrix(V_crop))
        print_polynomial_matrix("V_inv_crop", matrix(V_inv_crop))

    # λ^{d1→d2} = V_{[:,0..d2]} * (V^{-1})_{[0..d2,:]}
    return V_crop * V_inv_crop

## High-Level PSS Operations and Masked Arithmetic

This section implements the *algorithmic layer* of Packed Secret Sharing (PSS) on top of the precomputed linear maps.
It includes:

- **PSS encoding and decoding** (full and partial-share reconstruction),
- **Polynomial reconstruction and error detection** via Vandermonde interpolation,
- **Binary operations on masked polynomials**, and
- **LaOla-based masked multiplication and squaring**, including strong zero-encoding.

All routines here operate purely through linear maps and share-wise operations, and directly correspond to
Algorithm 9 and Section 4 of *“All-You-Can-Compute: Packed Secret Sharing for Combined Resilience”* (ePrint 2025/035).

In [14]:
# ============================================================
# Packed Secret Sharing (PSS): Encoding / Decoding (High-Level)
# ============================================================
#
# Conventions:
#   - k = num_secrets          (number of secrets embedded)
#   - n = num_shares           (number of shares)
#   - d = degree               (polynomial degree bound)
#   - shares_d_plus_one_minus_k = the (d+1-k) “randomness shares” that, together
#     with the k secrets, define the (d+1)-vector that is then mapped to n shares
#     via the precomputed encoding matrix M_enc.
#
# Shapes (as matrices/vectors over ff):
#   - M_enc: (n-(d+1-k)) x (d+1)
#   - M_dec: k x n
#
# Notes:
#   - We keep the API explicit for clarity and to match the paper notation.
#   - We validate basic parameter consistency early for easier debugging.

def pss_encoding(secrets_k, shares_d_plus_one_minus_k, M_enc, num_shares, degree, num_secrets):
    """
    Encode k secrets into n PSS shares using the precomputed encoding matrix M_enc.

    Input:
      - secrets_k: length k list (secret values)
      - shares_d_plus_one_minus_k: length (d+1-k) list (random shares / padding)
      - M_enc: encoding matrix mapping the (d+1)-vector -> remaining shares
      - num_shares: n
      - degree: d
      - num_secrets: k

    Output:
      - shares_n: length n list of share values
    """
    # --- sanity checks ---
    if len(secrets_k) != num_secrets:
        raise ValueError(f"secrets_k must have length k={num_secrets}, got {len(secrets_k)}")
    expected_pad = degree + 1 - num_secrets
    if len(shares_d_plus_one_minus_k) != expected_pad:
        raise ValueError(f"shares_d_plus_one_minus_k must have length d+1-k={expected_pad}, got {len(shares_d_plus_one_minus_k)}")
    if num_shares < degree + 1:
        raise ValueError(f"need n >= d+1 for encoding, got n={num_shares}, d={degree}")

    # Build the (d+1)-vector [secrets || padding] as a column vector.
    base_vec = matrix(secrets_k + shares_d_plus_one_minus_k).transpose()  # (d+1) x 1

    # Compute the remaining shares (those beyond the first (d+1-k) entries).
    # M_enc returns a column vector of length (n-(d+1-k)).
    shares_rest = M_enc * base_vec

    # Assemble full share vector:
    # - start with the first (d+1-k) values (the “given” part),
    # - append the computed remainder.
    shares_n = list(shares_d_plus_one_minus_k)
    for i in range(expected_pad, num_shares):
        shares_n.append(shares_rest[i - expected_pad, 0])

    return shares_n


def pss_decoding(shares_n, M_dec, num_shares=None):
    """
    Decode k secrets from n shares using the precomputed decoding matrix M_dec.

    Input:
      - shares_n: length n list
      - M_dec: decoding matrix (k x n)

    Output:
      - rec_secrets: column vector (k x 1) of reconstructed secrets
    """
    if num_shares is not None and len(shares_n) != num_shares:
        raise ValueError(f"shares_n must have length n={num_shares}, got {len(shares_n)}")

    rec_secrets = M_dec * matrix(shares_n).transpose()
    return rec_secrets


def pss_decoding_from_all_partial_shares_combinations(
    ff,
    shares_supports,
    secrets_supports,
    num_shares,
    num_secrets,
    shares_n,
    num_partial_shares,
):
    """
    Decode secrets from *all* combinations of 'num_partial_shares' shares.

    For each subset of supports of size num_partial_shares:
      - build a partial decoding matrix M_dec_partial
      - decode secrets from the corresponding partial share vector

    Returns a list of decoded secret vectors (one per subset).
    """
    if len(shares_n) != num_shares:
        raise ValueError(f"shares_n must have length n={num_shares}, got {len(shares_n)}")
    if not (1 <= num_partial_shares <= num_shares):
        raise ValueError(f"num_partial_shares must be in [1, n], got {num_partial_shares}")
    if len(shares_supports) < num_shares:
        raise ValueError("shares_supports shorter than num_shares")
    if len(secrets_supports) < num_secrets:
        raise ValueError("secrets_supports shorter than num_secrets")

    partial_shares_list, partial_supports_list = create_shares_supports_list_for_partial_shares(
        shares_n,
        shares_supports,
        num_shares,
        num_partial_shares,
    )

    rec_list = []
    for partial_vals, partial_supps in zip(partial_shares_list, partial_supports_list):
        M_dec_partial = generate_M_dec(
            ff,
            secrets_supports,
            partial_supps,
            num_partial_shares,
            num_secrets,
        )
        rec_list.append(pss_decoding(partial_vals, M_dec_partial, num_shares=num_partial_shares))

    return rec_list

In [ ]:
# ============================================================
# Polynomial Reconstruction and Error Detection
# ============================================================

# Reconstruct the polynomial coefficients from all 'n' shares, and check if the high order (> degree) coefficients are zeros
def error_detection(shares_n, shares_supports, num_shares, degree):
    mat_V_inv = generate_vandermonde(ff, shares_supports, num_shares, num_shares).inverse()
    coefficient_n = mat_V_inv * matrix(shares_n).transpose()
    print_polynomial_matrix("coefficient_n: ", matrix(coefficient_n))
    for d in range(degree + 1, num_shares):
        if coefficient_n[d] != 0:
            print("Error detected!")
            return False
    print("No error detected.")
    return True


# Reconstruct the polynomial coefficients from all n shares and compute the polynomial degree
def coefficients_reconstruction(ff, shares_n, shares_supports, num_shares):
    mat_V_inv = generate_vandermonde(ff, shares_supports, num_shares, num_shares).inverse()
    coefficient_n = mat_V_inv * matrix(shares_n).transpose()
    coefficient_n_list = []
    for i in range(num_shares):
        coefficient_n_list.append(coefficient_n[i, 0])

    # compute the degree of the polynomial by finding the location of the first zero coefficient
    list_zero_position = np.where(np.array(coefficient_n_list) != 0)
    degree_max = 0

    degree_max = list_zero_position[0][-1]

    return coefficient_n_list, degree_max

In [16]:
# ============================================================
# Binary Operations on Masked Polynomials
# ============================================================

# Binary multiplication algorithm without refresh (cf. Algorithm 9)
def binary_mult(f_prime_A, f_double_prime_A, f_prime_B, f_double_prime_B):
    f_prime_AB = sw_mult(f_prime_A, f_prime_B)
    f_prime_double_prime_AB = sw_mult(f_prime_A, f_double_prime_B)
    f_double_prime_prime_AB = sw_mult(f_double_prime_A, f_prime_B)
    f_double_prime_double_prime_AB = sw_mult(f_double_prime_A, f_double_prime_B)
    result = sw_add(sw_add(f_prime_AB, f_prime_double_prime_AB),
                    sw_add(f_double_prime_prime_AB, f_double_prime_double_prime_AB))
    return result

In [17]:
# ============================================================
# LaOla-Based Masked Multiplication and Squaring
# ============================================================

# LaOla multiplication
# Please check the paper (All-You-Can-Compute: Packed Secret Sharing for Combined Resilience (Algorithm 9), https://eprint.iacr.org/2025/035) for details
def binary_mult_LaOla(ff, support_point_range, shares_n_A, shares_n_B, secrets_supports, shares_supports, num_secrets,
                      num_shares, degree, A_tilde, lambda_hat_matrix, ceil_n_half, floor_n_half,
                      ceil_d_half, floor_d_half):
    f_prime_A, f_double_prime_A = split_red(ff, support_point_range, shares_n_A, shares_supports, secrets_supports,
                                            num_shares, degree, num_secrets, ceil_n_half, floor_n_half,
                                            floor_d_half, lambda_hat_matrix)
    f_prime_B, f_double_prime_B = split_red(ff, support_point_range, shares_n_B, shares_supports, secrets_supports,
                                            num_shares, degree, num_secrets, ceil_n_half, floor_n_half,
                                            ceil_d_half, lambda_hat_matrix)

    # generate strong zero-encodings
    optsZEnc_share_n = optsZEnc(ff, support_point_range, secrets_supports, shares_supports, num_secrets, num_shares,
                                degree, A_tilde)

    f_prime_AB = sw_mult(f_prime_A, f_prime_B)
    f_prime_double_prime_AB = sw_mult(f_prime_A, f_double_prime_B)
    f_double_prime_prime_AB = sw_mult(f_double_prime_A, f_prime_B)
    f_double_prime_double_prime_AB = sw_mult(f_double_prime_A, f_double_prime_B)

    result = sw_add(sw_add(sw_add(optsZEnc_share_n, f_prime_AB), f_prime_double_prime_AB),
                    sw_add(f_double_prime_prime_AB, f_double_prime_double_prime_AB))
    return result


# Square operation using LaOla multiplication, will be replaced with Frobenius based squaring operations
def poly_masked_square(ff, support_point_range, shares_n, secrets_supports, shares_supports, num_secrets, num_shares,
                       degree, A_tilde, lambda_hat_matrix, ceil_n_half, floor_n_half, ceil_d_half,
                       floor_d_half):
    square_result = binary_mult_LaOla(ff, support_point_range, shares_n, shares_n, secrets_supports, shares_supports,
                                      num_secrets, num_shares, degree, A_tilde, lambda_hat_matrix, 
                                      ceil_n_half, floor_n_half, ceil_d_half, floor_d_half)
    return square_result

In [18]:
# ------------------------------------------------------------
# Optimized zero-encoding (optZEnc) and strong optZEnc (optsZEnc)
# ------------------------------------------------------------

def optZEnc(
    ff,
    secrets_supports,
    shares_supports,
    num_secrets,
    num_shares,
    degree_optzenc,
    randoffset,
    random_poly_list,
    A_tilde=None,   # allow injecting precomputed A_tilde for speed
):
    """
    Optimized zero-encoding (cf. Algorithm 1).

    Produces an n-share encoding of k zeros with degree <= degree_optzenc,
    using two randomness-saving knobs:
      (1) randoffset: force the first `randoffset` share positions to zero
      (2) low-degree polynomial: reduce degree_optzenc (used by SplitRed)

    If A_tilde is not provided, it is computed from (n,k,degree_optzenc) and supports.
    """

    # --- sanity checks (use exceptions, not asserts) ---
    if degree_optzenc < num_secrets:
        raise ValueError("degree_optzenc must be >= num_secrets to embed k secrets.")
    if not (0 <= randoffset < degree_optzenc + 1 - num_secrets):
        raise ValueError(
            "randoffset must satisfy 0 <= randoffset < (degree_optzenc + 1 - num_secrets)."
        )

    needed_rand = (degree_optzenc + 1 - num_secrets) - randoffset
    if len(random_poly_list) != needed_rand:
        raise ValueError(f"random_poly_list must have length {needed_rand}, got {len(random_poly_list)}.")

    # --- precompute A_tilde once if not provided ---
    if A_tilde is None:
        M_enc_tmp = generate_M_enc(ff, secrets_supports, shares_supports, num_secrets, num_shares, degree_optzenc)
        A_tilde = generate_A_tilde(ff, M_enc_tmp, num_secrets, degree_optzenc)

    # --- initialize output shares ---
    res = [ff(0)] * num_shares

    # first randoffset entries are forced zero (already zero)
    # fill next (degree_optzenc + 1 - k - randoffset) entries with randomness
    for i in range(randoffset, degree_optzenc + 1 - num_secrets):
        res[i] = random_poly_list[i - randoffset]

    # compute remaining shares using A_tilde
    # NOTE: j runs only up to (degree_optzenc - num_secrets), not (degree_optzenc + 1 - num_secrets)
    # because A_tilde is defined for that range in the paper construction.
    j_stop = degree_optzenc - num_secrets + 1  # inclusive end in math, exclusive in range()
    for i in range(degree_optzenc + 1 - num_secrets, num_shares):
        acc = ff(0)
        for j in range(randoffset, j_stop):
            acc += res[j] * A_tilde[i][j]
        res[i] = acc

    return res


def optsZEnc(ff, support_point_range, secrets_supports, shares_supports, num_secrets, num_shares, degree_optszenc, A_tilde=None):
    """
    Optimized strong zero-encoding (cf. Algorithm 3).

    Sums optZEnc instances for randoffset = 0..(degree_optszenc - num_secrets).
    Precomputes A_tilde once for efficiency.
    """
    if degree_optszenc < num_secrets:
        raise ValueError("degree_optszenc must be >= num_secrets.")
    if num_shares <= 0:
        raise ValueError("num_shares must be positive.")

    # precompute A_tilde once if not provided
    if A_tilde is None:
        M_enc_tmp = generate_M_enc(ff, secrets_supports, shares_supports, num_secrets, num_shares, degree_optszenc)
        A_tilde = generate_A_tilde(ff, M_enc_tmp, num_secrets, degree_optszenc)

    res = [ff(0)] * num_shares
    max_randoffset = degree_optszenc - num_secrets  # inclusive

    for randoffset in range(0, max_randoffset + 1):
        needed_rand = (degree_optszenc + 1 - num_secrets) - randoffset
        rand_list = generate_random_field_elements(ff, support_point_range, needed_rand)
        res = sw_add(
            res,
            optZEnc(
                ff, secrets_supports, shares_supports,
                num_secrets, num_shares, degree_optszenc,
                randoffset, rand_list,
                A_tilde=A_tilde,
            ),
        )
    return res

In [19]:
# =============================================================================
# SplitRed (Algorithm 7, ePrint 2025/035): degree-splitting with masked re-randomization
# =============================================================================
# Goal
# ----
# Given an n-share vector f (evaluations of a degree-≤d polynomial embedding k secrets),
# compute two n-share vectors (f_prime, f_double_prime) such that:
#
#   (1) f_prime  + f_double_prime  ==  λ̂ · f   +   (zero-encoding terms)
#       and the resulting sharing behaves as required by Algorithm 7, and
#   (2) the construction enforces the intended degree behavior (degree reduction +
#       re-randomization) while preserving the embedded secrets.
#
# High-level idea
# ---------------
# SplitRed partitions indices into two halves:
#   - first half (size floor(n/2))  contributes to f_prime
#   - second half (size ceil(n/2))  contributes to f_double_prime
#
# For each index j, we form:
#   f_cal[j] = (λ̂ column j applied to f[j])  +  g[j]
# where:
#   - λ̂ ("lambda_hat_matrix") combines degree reduction and error propagation
#   - g[j] is a masking term built from optZEnc-based zero-encodings
#
# We then sum f_cal[j] over the first half to obtain f_prime,
# and sum f_cal[j] over the second half to obtain f_double_prime.
#
# Randomness-saving trick used here
# --------------------------------
# We generate many zero-encodings g_hat/g_tilde but vary their degree to reduce
# randomness consumption:
#   - for g_hat: degrees d, d-1, d-2, ... down to k, then fall back to d
#   - for g_tilde: degrees low_degree_optzenc, low_degree_optzenc-1, ... down to k,
#                  then fall back to low_degree_optzenc
#
# Security note: we set randoffset = 0 here (do NOT force leading shares to zero),
# because using randoffset>0 inside SplitRed can harm probing-security assumptions.
#
# Parameters / inputs
# -------------------
# ff                 : finite field
# support_point_range: sampling range for test randomness
# f                  : length-n share vector (field elements)
# shares_supports    : evaluation points for the n shares
# secrets_supports   : evaluation points where secrets are embedded
# num_shares (n)     : number of shares
# degree (d)         : input polynomial degree bound
# num_secrets (k)    : number of embedded secrets
# ceil_n_half        : ceil(n/2)
# floor_n_half       : floor(n/2)
# low_degree_optzenc : chosen low degree for the “low-degree” zero-encodings
# lambda_hat_matrix  : λ̂ (n×n), combines degree reduction + error propagation
#
# Output
# ------
# (f_prime, f_double_prime): two length-n share vectors
# =============================================================================

def split_red(ff, support_point_range, f, shares_supports, secrets_supports, num_shares, degree, num_secrets,
              ceil_n_half, floor_n_half, low_degree_optzenc, lambda_hat_matrix):
    num_faults = num_shares - 1 - degree

    # guarantee that after degree reduction, the polynomial with degree floor(degree) is sufficient to hide k secrets
    assert (floor(degree) >= num_secrets), (
        f"[SplitRed] invalid parameters: floor(degree) < num_secrets "
        f"(floor({degree})={floor(degree)}, k={num_secrets})."
    )

    # initialize all the intermediate shares
    g_hat = create_list_list(ceil_n_half, num_shares)
    g_tilde = create_list_list(num_shares, num_shares)
    g = create_list_list(num_shares, num_shares)
    f_prime_cal = create_list_list(num_shares, num_shares)
    f_cal = create_list_list(num_shares, num_shares)
    f_prime = [0] * num_shares
    f_double_prime = [0] * num_shares

    # we don't use the randoffset to save randomness in split_red as it may cause probing security issues
    randoffset = 0

    # This is the degree of the zero encoding with a high degree
    high_degree_optzenc = degree

    # line 2-3: build g_hat[j] (high-degree-ish zero-encodings, with degree-scheduling to save randomness)
    for j in range(ceil_n_half):
        degree_optzenc_tmp = high_degree_optzenc - j
        if degree_optzenc_tmp < num_secrets:
            degree_optzenc_tmp = high_degree_optzenc

        random_poly_list = generate_random_field_elements(
            ff, support_point_range,
            degree_optzenc_tmp + 1 - num_secrets - randoffset
        )
        g_hat[j] = optZEnc(
            ff, secrets_supports, shares_supports,
            num_secrets, num_shares,
            degree_optzenc_tmp, randoffset,
            random_poly_list
        )

    # line 4-6: build g_tilde[j] and g[j] for the first half
    for j in range(floor_n_half):
        degree_optzenc_tmp = low_degree_optzenc - j
        if degree_optzenc_tmp < num_secrets:
            degree_optzenc_tmp = low_degree_optzenc

        random_poly_list = generate_random_field_elements(
            ff, support_point_range,
            degree_optzenc_tmp + 1 - num_secrets - randoffset
        )
        g_tilde[j] = optZEnc(
            ff, secrets_supports, shares_supports,
            num_secrets, num_shares,
            degree_optzenc_tmp, randoffset,
            random_poly_list
        )
        g[j] = sw_add(g_tilde[j], g_hat[j])

    # line 7-8: odd-n correction
    if num_shares % 2 == 1:
        g[floor_n_half - 1] = sw_add(g[floor_n_half - 1], g_hat[ceil_n_half - 1])

    # line 9-11: λ̂ contribution for the first half
    for j in range(floor_n_half):
        for i in range(num_shares):
            f_prime_cal[j][i] = poly_mult(lambda_hat_matrix[i][j], f[j])

    # line 12-13: add zero-encodings for the first half
    for j in range(floor_n_half):
        f_cal[j] = sw_add(f_prime_cal[j], g[j])

    # line 14-15: sum first half -> f_prime
    for j in range(floor_n_half):
        f_prime = sw_add(f_prime, f_cal[j])

    # line 16-18: build g_tilde/g for the second half (paired with g_hat[j])
    for j in range(ceil_n_half):
        degree_optzenc_tmp = low_degree_optzenc - j
        if degree_optzenc_tmp < num_secrets:
            degree_optzenc_tmp = low_degree_optzenc

        random_poly_list = generate_random_field_elements(
            ff, support_point_range,
            degree_optzenc_tmp + 1 - num_secrets - randoffset
        )
        g_tilde[floor_n_half + j] = optZEnc(
            ff, secrets_supports, shares_supports,
            num_secrets, num_shares,
            degree_optzenc_tmp, randoffset,
            random_poly_list
        )
        g[floor_n_half + j] = sw_add(g_tilde[floor_n_half + j], g_hat[j])

    # line 19-21: λ̂ contribution for the second half
    for j in range(ceil_n_half):
        for i in range(num_shares):
            f_prime_cal[floor_n_half + j][i] = poly_mult(
                lambda_hat_matrix[i][floor_n_half + j],
                f[floor_n_half + j]
            )

    # line 22-23: add zero-encodings for the second half
    for j in range(ceil_n_half):
        f_cal[floor_n_half + j] = sw_add(f_prime_cal[floor_n_half + j], g[floor_n_half + j])

    # line 24-25: sum second half -> f_double_prime
    for j in range(ceil_n_half):
        f_double_prime = sw_add(f_double_prime, f_cal[floor_n_half + j])

    return f_prime, f_double_prime


In [20]:
# =============================================================================
# Masked AES S-box via polynomial evaluation (GF(2^8) inversion + affine layer)
# =============================================================================
# This routine computes the AES S-box on *packed secret shares* by evaluating the
# standard exponentiation chain for inversion in GF(2^8):
#
#     inv(x) = x^254   (with the convention 0 -> 0, inherited from the chain)
#
# using:
#   - poly_masked_square(...)  : masked squaring (implemented via LaOla multiplication here)
#   - binary_mult_LaOla(...)   : masked multiplication
#
# After inversion, it applies the AES affine transform (as a linearized polynomial)
# in the masked domain, using only:
#   - share-wise additions
#   - multiplication by public constants (unary_mult)
#
# Inputs/Outputs are length-n share vectors over ff.
# =============================================================================
def poly_masked_sbox(ff, support_point_range, f, shares_supports, secrets_supports, num_shares, degree, num_secrets,
                     ceil_n_half, floor_n_half, ceil_d_half, floor_d_half, A_tilde, lambda_hat_matrix):

    # ---------- basic sanity checks (helpful for debugging) ----------
    if len(f) != num_shares:
        raise ValueError(f"[poly_masked_sbox] len(f)={len(f)} != num_shares={num_shares}")

    # ---------- helper: dump a compact share vector in integer form ----------
    def _dump_vec(label, v, max_len=10):
        try:
            ints = [x.to_integer() if hasattr(x, "to_integer") else int(x) for x in v]
        except Exception:
            ints = [repr(x) for x in v]
        if len(ints) > max_len:
            head = ", ".join(map(str, ints[:max_len]))
            print(f"{label} (len={len(v)}): [{head}, ...]")
        else:
            print(f"{label} (len={len(v)}): {ints}")

    # ---------- helper: check degree bound and print reconstruction info ----------
    def _check_degree(label, shares, deg_bound):
        try:
            _, deg_max = coefficients_reconstruction(ff, shares, shares_supports, num_shares)
        except Exception as e:
            print(f"[poly_masked_sbox] degree reconstruction failed at {label}: {e!r}")
            _dump_vec(f"[poly_masked_sbox] shares snapshot at {label}", shares)
            raise
        if not (0 <= deg_max <= deg_bound):
            print(f"[poly_masked_sbox] DEGREE VIOLATION at {label}: deg_max={deg_max}, bound={deg_bound}")
            _dump_vec(f"[poly_masked_sbox] shares snapshot at {label}", shares)
            raise AssertionError(f"[poly_masked_sbox] degree violation at {label}: {deg_max} > {deg_bound}")
        return deg_max

    # ---------- helper: compute masked square/mult with local error context ----------
    def _sq(label, xshares):
        try:
            y = poly_masked_square(
                ff, support_point_range, xshares,
                secrets_supports, shares_supports, num_secrets, num_shares,
                degree, A_tilde, lambda_hat_matrix,
                ceil_n_half, floor_n_half, ceil_d_half, floor_d_half
            )
        except Exception as e:
            print(f"[poly_masked_sbox] ERROR in poly_masked_square at {label}: {e!r}")
            _dump_vec(f"[poly_masked_sbox] input shares to square at {label}", xshares)
            raise
        return y

    def _mul(label, xshares, yshares):
        try:
            z = binary_mult_LaOla(
                ff, support_point_range, xshares, yshares,
                secrets_supports, shares_supports, num_secrets, num_shares,
                degree, A_tilde, lambda_hat_matrix,
                ceil_n_half, floor_n_half, ceil_d_half, floor_d_half
            )
        except Exception as e:
            print(f"[poly_masked_sbox] ERROR in binary_mult_LaOla at {label}: {e!r}")
            _dump_vec(f"[poly_masked_sbox] input A shares to mult at {label}", xshares)
            _dump_vec(f"[poly_masked_sbox] input B shares to mult at {label}", yshares)
            raise
        return z

    # ---------- allocate vectors ----------
    res = [ff(0)] * num_shares
    tmp = [ff(0)] * num_shares

    # ======================================================================
    # Part 1: masked inversion via exponentiation chain (x^254)
    # ======================================================================
    # x^2, x^4, x^8
    f_two   = _sq("inv:x^2",  f)
    f_four  = _sq("inv:x^4",  f_two)
    f_eight = _sq("inv:x^8",  f_four)

    # x^9 = x^8 * x
    f_nine  = _mul("inv:x^9 = x^8*x", f_eight, f)

    # x^18 = (x^9)^2
    f_eighteen = _sq("inv:x^18", f_nine)

    # x^19 = x^18 * x
    f_nineteen = _mul("inv:x^19 = x^18*x", f_eighteen, f)

    # x^36 = (x^18)^2
    f_thirtysix = _sq("inv:x^36", f_eighteen)

    # x^55 = x^36 * x^19
    f_fiftyfive = _mul("inv:x^55 = x^36*x^19", f_thirtysix, f_nineteen)

    # x^72 = (x^36)^2
    f_seventytwo = _sq("inv:x^72", f_thirtysix)

    # x^127 = x^72 * x^55
    f_onehundrettwentyseven = _mul("inv:x^127 = x^72*x^55", f_seventytwo, f_fiftyfive)

    # x^254 = (x^127)^2
    f_twohundretfiftyfour = _sq("inv:x^254", f_onehundrettwentyseven)

    # Optional debug: ensure degree stayed within expected bound (same degree bound as gadget)
    _check_degree("after inversion x^254", f_twohundretfiftyfour, degree)

    # ======================================================================
    # Part 2: AES affine transform (linearized polynomial), masked domain
    # ======================================================================
    # We now set f := inv(x) = x^254 and compute:
    #   S(x) = 0x63 ⊕ 0x05·f ⊕ 0x09·f^2 ⊕ 0xF9·f^4 ⊕ 0x25·f^8 ⊕ 0xF4·f^16 ⊕ 0x01·f^32 ⊕ 0xB5·f^64 ⊕ 0x8F·f^128
    #
    # All coefficients are public constants; multiplication is share-wise by constants.

    f = f_twohundretfiftyfour

    # public constants replicated to length-n share vectors
    hex63 = int_to_polylist(ff,  99, num_shares)
    hex05 = int_to_polylist(ff,   5, num_shares)
    hex09 = int_to_polylist(ff,   9, num_shares)
    hexf9 = int_to_polylist(ff, 249, num_shares)
    hex25 = int_to_polylist(ff,  37, num_shares)
    hexf4 = int_to_polylist(ff, 244, num_shares)
    hex01 = int_to_polylist(ff,   1, num_shares)
    hexb5 = int_to_polylist(ff, 181, num_shares)
    hex8f = int_to_polylist(ff, 143, num_shares)

    # res = 0x63 + 0x05*f
    res = unary_add(res, hex63)
    tmp = unary_mult(f, hex05)
    res = sw_add(res, tmp)

    # add 0x09*f^2
    f_two = _sq("aff:f^2", f)
    tmp = unary_mult(f_two, hex09)
    res = sw_add(res, tmp)

    # add 0xF9*f^4
    f_four = _sq("aff:f^4", f_two)
    tmp = unary_mult(f_four, hexf9)
    res = sw_add(res, tmp)

    # add 0x25*f^8
    f_eight = _sq("aff:f^8", f_four)
    tmp = unary_mult(f_eight, hex25)
    res = sw_add(res, tmp)

    # add 0xF4*f^16
    f_sixteen = _sq("aff:f^16", f_eight)
    tmp = unary_mult(f_sixteen, hexf4)
    res = sw_add(res, tmp)

    # add 0x01*f^32
    f_thirtytwo = _sq("aff:f^32", f_sixteen)
    tmp = unary_mult(f_thirtytwo, hex01)
    res = sw_add(res, tmp)

    # add 0xB5*f^64
    f_sixtyfour = _sq("aff:f^64", f_thirtytwo)
    tmp = unary_mult(f_sixtyfour, hexb5)
    res = sw_add(res, tmp)

    # add 0x8F*f^128
    f_onehundrettwentyeight = _sq("aff:f^128", f_sixtyfour)
    tmp = unary_mult(f_onehundrettwentyeight, hex8f)
    res = sw_add(res, tmp)

    # Final sanity: degree bound should still be within "degree"
    _check_degree("final sbox shares", res, degree)

    return res

In [21]:
# -----------------------------------------------------------------------------
# Zero-encodings R^{(0)}, ..., R^{(e-1)} for constructing A_hat (Shamir case)
# -----------------------------------------------------------------------------
#
# This function generates the family of special zero-encodings R^{(k)} used in
# the construction of the matrix A_hat for the adapted SplitRed gadget.
#
# Setting:
#   - Shamir secret sharing (secrets_supports == shares_supports)
#   - n = num_shares
#   - d = indegree
#   - outdegree = floor(d/2)
#   - e = n - d - 1  (maximum number of faults)
#
# Goal:
#   For each k ∈ {0, ..., e-1}, construct an n-share vector R^{(k)} such that
#   the underlying polynomial r^{(k)}(x) satisfies:
#
#     • coeff_0            : random
#     • coeff_1 ... coeff_outdegree : 0
#     • coeff_{outdegree+1} ... coeff_{n-1} : random
#
#   In other words, only the constant term and the high-degree coefficients
#   (strictly above outdegree) are randomized, while all low-degree coefficients
#   that could influence the embedded secrets are zero.
#
# Intuition:
#   - These R^{(k)} encodings inject fresh randomness *only* into the high-degree
#     part of the reconstructed polynomial.
#   - When lifted into A_hat, they ensure that any non-zero high-degree term
#     (e.g., caused by faults) forces a re-randomization of the secrets.
#
# Randomness layout:
#   - Each R^{(k)} consumes (num_shares - outdegree - 1) random field elements:
#       * 1 value for coeff_0
#       * (num_shares - outdegree - 1) values for coeff_{outdegree+1} ... coeff_{n-1}
#   - random_poly_list is interpreted as a flat list, sliced per k.
#
# Parameters:
#   ff              : finite field
#   V               : Vandermonde matrix on share support points (n × n)
#   num_shares      : n
#   indegree        : d (input polynomial degree)
#   outdegree       : floor(d/2)
#   random_poly_list: list of random field elements of length
#                     e * (num_shares - outdegree - 1)
#
# Returns:
#   ZEnc_R_list:
#     A list of e zero-encodings, where each entry is a length-n list of shares
#     representing R^{(k)} evaluated at the share support points.
#
# Notes:
#   - This function operates purely in the share domain using the Vandermonde
#     matrix V; no interpolation is performed here.
#   - Correctness is typically verified by reconstructing coefficients via V^{-1}
#     (see test_generate_ZEnc_R_for_A_hat_shamir).
# -----------------------------------------------------------------------------

def generate_ZEnc_R_for_A_hat_shamir(ff, V, num_shares, indegree, outdegree, random_poly_list):
    # The number of faults the attacker can inject
    num_faults = num_shares-indegree-1
    
    # generate 'e=n+1-d' special type of zero-encodings R, where only the 0, d/2+1, ..., n-1 coefficients are random values, the rest coefficients are zeros
    
    # list for 'e' zero-encodings, R^{(0)}, ..., R^{(e-1)}
    ZEnc_R_list=[]
    for k in range(num_faults):

        # the list for the shares of the k-th zero-encoding R^{(k)}, k \in {0, ..., e-1}
        ZEnc_R_k_share_list=[0] * num_shares

        # 0-th coefficient for n shares of R^{(k)}
        for i in range(num_shares):
            ZEnc_R_k_share_list[i]= random_poly_list[k*(num_shares-outdegree-1)+0]
        
        # (d/2+1, ..., n-1)-th coefficients for n shares of R^{(k)}
        for j in range(outdegree+1, num_shares):
            for i in range(num_shares):
                ZEnc_R_k_share_list[i]= ZEnc_R_k_share_list[i]+poly_mult(random_poly_list[k*(num_shares-outdegree-1)+(j-outdegree)],V[i,j])
        ZEnc_R_list.append(ZEnc_R_k_share_list)
    return ZEnc_R_list                

In [22]:
# -----------------------------------------------------------------------------
# Construction of A_hat for the adapted SplitRed gadget (Shamir case)
# -----------------------------------------------------------------------------
#
# This function constructs the matrix A_hat used in the adapted SplitRed gadget
# (cf. Section 4.2 and Appendix A of
#   "All-You-Can-Compute: Packed Secret Sharing for Combined Resilience",
#   ePrint 2025/035).
#
# Setting:
#   - Shamir secret sharing (secrets_supports == shares_supports)
#   - n = num_shares
#   - d = indegree
#   - outdegree = floor(d/2)
#   - e = n - d - 1  (maximum number of faults)
#
# Purpose:
#   A_hat combines two effects:
#
#     (1) Degree reduction:
#         It maps an n-share vector encoding a polynomial of degree ≤ d
#         to an n-share vector encoding a polynomial of degree ≤ outdegree,
#         while preserving the embedded secret(s).
#
#     (2) High-degree re-randomization (fault amplification):
#         If any high-degree coefficient (degrees d+1 .. n-1) is non-zero
#         (e.g., due to injected faults), the secret is re-randomized via
#         specially crafted zero-encodings.
#
#   Formally:
#       A_hat = λ^{(d → outdegree)}  +  R_hat
#
#   where:
#     - λ^{(d → outdegree)} is the standard Shamir degree-reduction matrix,
#     - R_hat injects randomness derived from high-degree coefficients only.
#
# Inputs:
#   ff              : finite field
#   V               : Vandermonde matrix on share support points (n × n),
#                     V[i,j] = x_i^j
#   V_inv           : inverse of V
#   num_shares      : n
#   indegree        : d (input polynomial degree)
#   outdegree       : floor(d/2)
#   random_poly_list: randomness consumed by the zero-encodings R^{(k)}
#
# Output:
#   matrix_A_hat_shamir:
#     An n × n matrix over ff implementing the A_hat linear map.
#
# Notes:
#   - All operations are linear and depend only on public parameters and
#     sampled randomness.
#   - The zero-encodings R^{(k)} are constructed such that they randomize
#     only the high-degree part of the polynomial representation.
# -----------------------------------------------------------------------------

def generate_A_hat_shamir(ff, V, V_inv, num_shares, indegree, outdegree, random_poly_list):
    
    # generate 'e=n+1-d' special type of zero-encodings R, where only the 0, d/2+1, ..., n-1 coefficients are random values, the rest coefficients are zeros
    ZEnc_R_list=generate_ZEnc_R_for_A_hat_shamir(ff, V, num_shares, indegree, outdegree, random_poly_list)

    # generate the R_hat matrix
    matrix_R_hat=zero_matrix(ff,num_shares, num_shares)
    for i in range(num_shares):
        for j in range(num_shares):
            for k in range(indegree+1, num_shares):
                matrix_R_hat[i,j] = poly_mult(ZEnc_R_list[k-indegree-1][i], V_inv[k,j])
                # matrix_R_hat[i,j] += poly_mult(ZEnc_R_list[k-indegree-1][i], V_inv[k,j])
    
    # generate the lambda^{(indegree -> outdegree)} matrix
    lambda_degree_reduction_shamir = generate_lambda_degree_reduction_shamir(ff, V, V_inv, num_shares, indegree, outdegree)

    matrix_A_hat_shamir = lambda_degree_reduction_shamir + matrix_R_hat

    return matrix_A_hat_shamir

## Functional Correctness and Robustness Validation Suite

This section implements a **comprehensive validation suite** for the executable
artifact accompanying *“UP TO 50% OFF: Efficient Implementation of Polynomial Masking.”*

The goal of this suite is to **systematically verify that the implemented
constructions satisfy their functional, algebraic, and robustness guarantees**
under the combined ((t, e))-adversary model assumed in the paper.

### Scope of validation

The tests in this section collectively cover the following aspects:

* **Functional correctness**
  All core primitives (encoding/decoding, zero-encodings, optimized zero-encodings,
  SplitRed, LaOla-based multiplication, and masked AES S-box evaluation) are verified
  to preserve and correctly reconstruct the embedded secrets.

* **Degree bounds and invariants**
  Each test explicitly checks that polynomial degrees remain within the bounds
  prescribed by the theory, both before and after degree reduction, masking,
  multiplication, and squaring.

* **Robustness against injected faults**
  Faults are injected directly into the share domain, and the tests verify that:

  * high-degree components appear as predicted by the theory,
  * error-propagation mechanisms (e.g., (E), (\hat{\lambda}), and (\hat{A}))
    correctly amplify or randomize these components, and
  * secrets remain protected and reconstructable.

* **Threshold and partial reconstruction guarantees**
  Decoding is validated not only from all (n) shares, but also from *all qualifying
  subsets* of shares, ensuring consistency with the underlying Shamir/PSS threshold
  properties.

### Structure and interpretation

The validation suite is organized as a sequence of **self-contained test functions**,
each targeting a specific property or gadget. Tests are designed to be:

* **Executable and reproducible**:
  All randomness is explicitly sampled within the notebook, and failing cases emit
  detailed diagnostic information (shares, coefficients, degrees, and reconstruction
  results).

* **Robust to probabilistic effects**:
  Some checks (in particular those related to fault randomization) are probabilistic
  by nature. Occasional rare failures are therefore expected and treated as
  informative events rather than silent errors.

Overall, this section should be read as an **end-to-end validation framework** that
demonstrates the correctness and robustness of the full implementation stack, rather
than as a collection of isolated unit tests.


In [23]:
def test_optZEnc_randoffset(ff, support_point_range, secrets_supports, shares_supports,
                            num_secrets, num_shares, degree_optzenc,
                            randoffset, M_dec):
    """
    Test purpose
    ------------
    Validate the correctness of optZEnc when the randoffset optimization
    is enabled, i.e., when the first randoffset output shares are forced to
    zero in order to save randomness.
    
    This test checks that enforcing zero-valued shares does **not** break:
      - the zero-encoding property,
      - the polynomial degree bound, or
      - the secret reconstruction guarantees.
    
    In other words, it ensures that `randoffset` is a *safe randomness-saving
    optimization*.
    
    What is being tested (high-level)
    ---------------------------------
    Given parameters (n, k, degree_optzenc):
    
    The function verifies that `optZEnc` produces an n-share encoding such that:
      1) The first `randoffset` shares are exactly zero.
      2) The reconstructed polynomial degree is ≤ `degree_optzenc`.
      3) The encoded secrets are all zeros when decoding from:
         - all n shares, and
         - any subset of (degree_optzenc + 1) shares.
    
    This matches the correctness requirements of optimized zero-encodings
    described in the paper.
    
    How the test works (step-by-step)
    ---------------------------------
    1) Randomness generation:
       - Samples exactly (degree_optzenc + 1 − num_secrets − randoffset)
         random field elements.
       - This corresponds to the reduced randomness budget enabled by
         the `randoffset` optimization.
    
    2) Zero-encoding construction:
       - Calls `optZEnc(...)` to generate `optZEnc_share_n`, a list of
         `num_shares` field elements encoding k = `num_secrets` zeros.
    
    3) Forced-zero share check:
       - Verifies that the first `randoffset` shares are exactly zero.
       - This ensures that optZEnc respects the randoffset constraint.
    
    4) Degree bound verification:
       - Reconstructs the polynomial coefficients from all n shares.
       - Computes the maximum non-zero coefficient index `degree_max`.
       - Asserts `degree_max ≤ degree_optzenc`.
    
    5) Full decoding correctness:
       - Decodes secrets from all n shares using `M_dec`.
       - Asserts that all decoded secrets are zero.
    
    6) Partial decoding correctness:
       - Decodes secrets from every subset of size (degree_optzenc + 1).
       - Asserts that all decoded secrets are zero for every subset.
       - This confirms correct threshold reconstruction behavior.
    
    Failure diagnostics
    -------------------
    On assertion failure, the test prints detailed debugging information:
      - Parameter values (n, k, degree_optzenc, randoffset),
      - The first few output shares,
      - The reconstructed polynomial degree,
      - Indices of non-zero coefficients,
      - The tail of the reconstructed coefficient vector,
      - Decoded secrets from full reconstruction (if available).
    
    The assertion error is then re-raised to halt the test.
    
    Returns
    -------
    True
        If all correctness checks pass.
    
    Raises
    ------
    AssertionError
        If any of the correctness conditions is violated.
    """
    
    # --- tiny local helper (inside function; not an extra global function) ---
    to_int = lambda x: (x.to_integer() if hasattr(x, "to_integer") else int(x))

    try:
        # Generate randomness for the zero-encodings
        random_poly_list = generate_random_field_elements(
            ff, support_point_range, degree_optzenc + 1 - num_secrets - randoffset
        )

        optZEnc_share_n = optZEnc(
            ff, secrets_supports, shares_supports,
            num_secrets, num_shares, degree_optzenc, randoffset, random_poly_list
        )

        # check if the first 'randoffset' shares are equal to zeros
        for i in range(randoffset):
            assert optZEnc_share_n[i] == 0, (
                f"first randoffset shares must be 0, but share[{i}]={to_int(optZEnc_share_n[i])}"
            )

        # reconstruct polynomial degree from n shares
        coefficient_n_list, degree_max = coefficients_reconstruction(
            ff, optZEnc_share_n, shares_supports, num_shares
        )
        assert 0 <= degree_max <= degree_optzenc, (
            f"degree_max out of range: degree_max={degree_max}, expected <= {degree_optzenc}"
        )

        # reconstruct the secrets (zeros) from all n shares
        rec_zeros = pss_decoding(optZEnc_share_n, M_dec)
        for i in range(num_secrets):
            assert rec_zeros[i] == 0, (
                f"decoded secret not zero at i={i}: rec_zeros[{i}]={to_int(rec_zeros[i])}"
            )

        # reconstruct the secrets from any 'degree_optzenc+1' shares
        rec_zeros_list = pss_decoding_from_all_partial_shares_combinations(
            ff, shares_supports, secrets_supports,
            num_shares, num_secrets, optZEnc_share_n, degree_optzenc + 1
        )
        for j in range(len(rec_zeros_list)):
            for i in range(num_secrets):
                assert rec_zeros_list[j][i] == 0, (
                    f"partial decoding not zero: subset#{j}, secret#{i}, "
                    f"value={to_int(rec_zeros_list[j][i])}"
                )

        return True

    except AssertionError as e:
        # --- dump detailed context for debugging ---
        print("\n[ASSERT FAIL] test_optZEnc_randoffset")
        print(f"  params: n={num_shares}, k={num_secrets}, degree_optzenc={degree_optzenc}, randoffset={randoffset}")

        # best-effort: show first few shares + degree info if available
        try:
            print("  optZEnc_share_n (first 10 ints):",
                  [to_int(x) for x in optZEnc_share_n[:min(10, len(optZEnc_share_n))]])
        except Exception as _:
            print("  optZEnc_share_n not available (failed before generation).")

        try:
            print(f"  reconstructed degree_max={degree_max}")
            # show where coefficients are non-zero (useful to see unexpected high-degree terms)
            nz = [idx for idx, c in enumerate(coefficient_n_list) if c != 0]
            print("  nonzero coefficient indices:", nz[:50], ("..." if len(nz) > 50 else ""))
            # print tail coefficients (often where bugs show up)
            tail = coefficient_n_list[max(0, num_shares-10):]
            print("  last coefficients (ints):", [to_int(x) for x in tail])
        except Exception:
            pass

        # print decoded zeros if available
        try:
            print("  rec_zeros (ints):", [to_int(rec_zeros[i]) for i in range(min(num_secrets, len(rec_zeros)))])
        except Exception:
            pass

        # print assertion message
        print("  AssertionError:", str(e) if str(e) else "(no message)")
        raise

In [24]:
def test_optZEnc_degree(
    ff,
    support_point_range,
    secrets_supports,
    shares_supports,
    num_secrets,
    num_shares,
    degree_optzenc,
    randoffset,
    M_dec,
):
    """
    Test purpose
    ------------
    Validate the correctness of optZEnc (optimized zero-encoding) when generating
    a low-degree zero-encoding of degree ≤ degree_optzenc.
    
    In this setting, `optZEnc` should output an n-share vector that:
      1) Encodes **k zeros** as the embedded secrets,
      2) Has reconstructed polynomial degree bounded by `degree_optzenc`,
      3) Optionally forces the first `randoffset` shares to be exactly zero,
      4) Decodes to all-zero secrets both from:
         - all n shares (full decode), and
         - any subset of size (degree_optzenc + 1) shares (threshold property).
    
    What is being tested (high-level)
    ---------------------------------
    This is a *functional correctness* test for zero-encodings:
      - "Zero-encoding" means the shared polynomial embeds `num_secrets` zeros at
        `secrets_supports`, while remaining degrees are randomized.
      - Using a smaller `degree_optzenc` should reduce randomness usage while
        preserving correctness and decodability.
      - `randoffset` is additionally validated as a constraint on the *share values*
        (not just on decoded secrets).
    
    How the test works (step-by-step)
    ---------------------------------
    1) Randomness generation:
       - Samples exactly (degree_optzenc + 1 - num_secrets - randoffset) random field
         elements, which matches optZEnc’s expected randomness budget for this configuration.
    
    2) Zero-encoding generation:
       - Calls `optZEnc(...)` to produce `optZEnc_share_n`, a list of `num_shares`
         field elements representing shares of a polynomial intended to encode zeros.
    
    3) randoffset share constraint:
       - Verifies that the first `randoffset` output shares are exactly 0.
       - This checks the “forced-zero shares” feature of optZEnc.
    
    4) Degree bound check:
       - Reconstructs the polynomial coefficients from the n shares and computes
         the maximum non-zero coefficient index `degree_max`.
       - Asserts `degree_max <= degree_optzenc` to ensure the output really is a
         low-degree zero-encoding.
    
    5) Full decoding correctness:
       - Uses the provided decoding matrix `M_dec` to decode the embedded secrets
         from all n shares.
       - Asserts all decoded secrets are 0.
    
    6) Partial decoding correctness:
       - Enumerates all subsets of size (degree_optzenc + 1) shares and decodes
         the secrets from each subset.
       - Asserts all decoded secrets are 0 for every subset, confirming that the
         encoding is consistent with the intended reconstruction threshold.
    
    Failure diagnostics
    -------------------
    If any assertion fails, the test prints a structured debug dump:
      - Parameters (n, k, degree_optzenc, randoffset),
      - The randomness length actually used,
      - First few shares (as integers),
      - Reconstructed degree and indices of non-zero coefficients,
      - Decoded secrets from the full decoding.
    Then it re-raises the assertion error to stop the test harness.
    
    Returns
    -------
    True if all checks pass; otherwise raises AssertionError with diagnostics.
    """
    
    # small local helper (no extra global function)
    to_int = lambda x: (x.to_integer() if hasattr(x, "to_integer") else int(x))

    try:
        # ============================================================
        # 1. Generate zero-encoding
        # ============================================================
        random_poly_list = generate_random_field_elements(
            ff,
            support_point_range,
            degree_optzenc + 1 - num_secrets - randoffset,
        )

        optZEnc_share_n = optZEnc(
            ff,
            secrets_supports,
            shares_supports,
            num_secrets,
            num_shares,
            degree_optzenc,
            randoffset,
            random_poly_list,
        )

        # ============================================================
        # 2. Check randoffset shares are zero
        # ============================================================
        for i in range(randoffset):
            assert optZEnc_share_n[i] == 0, (
                f"randoffset violation: share[{i}] != 0, "
                f"value={to_int(optZEnc_share_n[i])}"
            )

        # ============================================================
        # 3. Degree check
        # ============================================================
        coefficient_n_list, degree_max = coefficients_reconstruction(
            ff,
            optZEnc_share_n,
            shares_supports,
            num_shares,
        )

        assert 0 <= degree_max <= degree_optzenc, (
            f"degree out of range: degree_max={degree_max}, "
            f"expected <= {degree_optzenc}"
        )

        # ============================================================
        # 4. Decode from all n shares (must be zero secrets)
        # ============================================================
        rec_zeros = pss_decoding(optZEnc_share_n, M_dec)
        for i in range(num_secrets):
            assert rec_zeros[i] == 0, (
                f"decoded secret not zero (full): "
                f"i={i}, value={to_int(rec_zeros[i])}"
            )

        # ============================================================
        # 5. Decode from any degree_optzenc+1 shares
        # ============================================================
        rec_zeros_list = pss_decoding_from_all_partial_shares_combinations(
            ff,
            shares_supports,
            secrets_supports,
            num_shares,
            num_secrets,
            optZEnc_share_n,
            degree_optzenc + 1,
        )

        for j, rec in enumerate(rec_zeros_list):
            for i in range(num_secrets):
                assert rec[i] == 0, (
                    f"decoded secret not zero (partial): "
                    f"subset={j}, i={i}, value={to_int(rec[i])}"
                )

        return True

    except AssertionError as e:
        # ============================================================
        # Detailed diagnostic dump
        # ============================================================
        print("\n[ASSERT FAIL] test_optZEnc_degree")
        print("  ---- parameters ----")
        print(f"  n={num_shares}, k={num_secrets}")
        print(f"  degree_optzenc={degree_optzenc}, randoffset={randoffset}")
        print(f"  randomness_len={len(random_poly_list)}")

        try:
            print("  ---- shares (first 10) ----")
            print([to_int(x) for x in optZEnc_share_n[:min(10, num_shares)]])
        except Exception:
            print("  shares not available")

        try:
            print(f"  reconstructed degree_max = {degree_max}")
            nz = [i for i, c in enumerate(coefficient_n_list) if c != 0]
            print("  non-zero coefficient indices:", nz[:20], "..." if len(nz) > 20 else "")
        except Exception:
            pass

        try:
            print("  decoded secrets (full):",
                  [to_int(rec_zeros[i]) for i in range(min(num_secrets, len(rec_zeros)))])
        except Exception:
            pass

        print("  AssertionError:", str(e) if str(e) else "(no message)")
        raise

In [25]:
def test_optsZEnc(
    ff, support_point_range,
    secrets_supports, shares_supports,
    num_secrets, num_shares,
    degree_optszenc,
    M_dec, A_tilde
):
    """
    Test purpose
    ------------
    Validate the correctness of strong optimized zero-encoding (optsZEnc).
    
    This test checks that `optsZEnc` produces a valid n-share encoding of
    `num_secrets` zeros with polynomial degree bounded by `degree_optszenc`,
    and that the encoded zeros can be correctly reconstructed from both:
      - all n shares, and
      - any subset of (degree_optszenc + 1) shares.
    
    Compared to `optZEnc`, `optsZEnc` is the *strong* variant used in higher-level
    gadgets (e.g., SplitRed and LaOla multiplication). It relies on a precomputed
    matrix `A_tilde` and does not expose low-level randomness parameters such as
    `randoffset`.
    
    What is being tested (high-level)
    ---------------------------------
    The function verifies that `optsZEnc` satisfies the defining properties of
    a zero-encoding:
    
      1) **Degree bound**:
         The reconstructed polynomial has degree ≤ `degree_optszenc`.
    
      2) **Zero secret correctness (full reconstruction)**:
         Decoding from all n shares yields `num_secrets` zeros.
    
      3) **Zero secret correctness (threshold reconstruction)**:
         Decoding from any subset of size (degree_optszenc + 1) also yields
         `num_secrets` zeros.
    
    Together, these checks confirm that `optsZEnc` is a correct zero-encoding
    gadget suitable for use inside fault-robust and degree-sensitive masked
    arithmetic constructions.
    
    How the test works (step-by-step)
    ---------------------------------
    1) Zero-encoding generation:
       - Calls `optsZEnc(...)` to generate an n-share vector
         `optsZEnc_share_n` encoding zeros.
    
    2) Degree verification:
       - Reconstructs the polynomial coefficients from the n shares.
       - Computes the maximum non-zero coefficient index `degree_max`.
       - Asserts `degree_max ≤ degree_optszenc`.
    
    3) Full decoding check:
       - Uses `M_dec` to decode secrets from all n shares.
       - Asserts that all decoded secrets are zero.
    
    4) Partial decoding check:
       - Decodes secrets from every subset of size (degree_optszenc + 1).
       - Asserts that all decoded secrets are zero for every subset.
       - This confirms correct threshold behavior.
    
    Failure diagnostics
    -------------------
    On assertion failure, the test prints detailed debugging information:
      - Parameters (n, k, degree_optszenc),
      - The first few output shares (integer form),
      - The reconstructed polynomial degree,
      - Indices of non-zero coefficients,
      - The tail of the reconstructed coefficient vector,
      - Decoded secrets from full reconstruction (if available).
    
    The assertion error is then re-raised to halt the test suite.
    
    Returns
    -------
    True
        If all correctness checks pass.
    
    Raises
    ------
    AssertionError
        If any correctness condition is violated.
    """
    
    # local helper only inside this function (not an extra global function)
    to_int = lambda x: (x.to_integer() if hasattr(x, "to_integer") else int(x))

    try:
        optsZEnc_share_n = optsZEnc(
            ff, support_point_range,
            secrets_supports, shares_supports,
            num_secrets, num_shares,
            degree_optszenc, A_tilde
        )

        # 1) degree check
        coefficient_n_list, degree_max = coefficients_reconstruction(
            ff, optsZEnc_share_n, shares_supports, num_shares
        )
        assert 0 <= degree_max <= degree_optszenc, (
            f"degree_max out of range: degree_max={degree_max}, expected <= {degree_optszenc}"
        )

        # 2) decode all-n should be zeros
        rec_zeros = pss_decoding(optsZEnc_share_n, M_dec)
        for i in range(num_secrets):
            assert rec_zeros[i] == 0, (
                f"decoded secret not zero (full): i={i}, value={to_int(rec_zeros[i])}"
            )

        # 3) decode any (degree_optszenc+1) should be zeros
        rec_zeros_list = pss_decoding_from_all_partial_shares_combinations(
            ff,
            shares_supports, secrets_supports,
            num_shares, num_secrets,
            optsZEnc_share_n,
            degree_optszenc + 1
        )
        for j in range(len(rec_zeros_list)):
            for i in range(num_secrets):
                assert rec_zeros_list[j][i] == 0, (
                    f"decoded secret not zero (partial): subset={j}, i={i}, value={to_int(rec_zeros_list[j][i])}"
                )

        return True

    except AssertionError as e:
        # ---- dump detailed context for debugging ----
        print("\n[ASSERT FAIL] test_optsZEnc")
        print("  ---- parameters ----")
        print(f"  n={num_shares}, k={num_secrets}, degree_optszenc={degree_optszenc}")

        # shares preview
        try:
            print("  ---- optsZEnc_share_n (first 10) ----")
            print([to_int(x) for x in optsZEnc_share_n[:min(10, len(optsZEnc_share_n))]])
        except Exception:
            print("  optsZEnc_share_n not available (failed before generation).")

        # degree + coefficients info
        try:
            print(f"  reconstructed degree_max = {degree_max}")
            nz = [idx for idx, c in enumerate(coefficient_n_list) if c != 0]
            print("  nonzero coefficient indices:", nz[:30], ("..." if len(nz) > 30 else ""))
            tail = coefficient_n_list[max(0, len(coefficient_n_list) - 10):]
            print("  last coefficients (ints):", [to_int(x) for x in tail])
        except Exception:
            pass

        # decoded zeros preview
        try:
            print("  decoded secrets (full, ints):",
                  [to_int(rec_zeros[i]) for i in range(min(num_secrets, len(rec_zeros)))])
        except Exception:
            pass

        print("  AssertionError:", str(e) if str(e) else "(no message)")
        raise

In [26]:
def test_M_lambda(
    ff, support_point_range,
    secrets_supports, shares_supports,
    num_secrets, num_shares,
    indegree, outdegree,
    M_enc, M_dec, V,
    ceil_n_half, floor_n_half, ceil_d_half, floor_d_half,
):

    """
    Test correctness of the degree-reduction linear map M_lambda.
    
    What this test validates
    ------------------------
    This routine checks that `generate_M_lambda(...)` correctly maps an n-share
    encoding of a degree-≤indegree polynomial to an n-share encoding of a
    degree-≤outdegree polynomial **while preserving the embedded secrets**.
    
    Concretely, it verifies three properties:
    
      1) Secret preservation (full reconstruction):
         After applying M_lambda and re-evaluating to shares, decoding from all n
         shares recovers exactly the original `k=num_secrets` secrets.
    
      2) Threshold correctness (partial reconstruction):
         Decoding from *any* subset of size (outdegree+1) shares also recovers the
         same secrets (i.e., the reduced-degree sharing has reconstruction threshold
         outdegree+1).
    
      3) Degree bound:
         The reconstructed polynomial degree from the output shares satisfies
         0 < degree_max <= outdegree.
    
         NOTE: This enforces a *strictly positive* degree. If your construction
         legitimately allows the reduced polynomial to be constant (degree_max == 0),
         change the check to: 0 <= degree_max <= outdegree.
    
    Inputs / expected environment
    -----------------------------
    - ff: finite field (Sage).
    - support_point_range: candidate integers for random sampling (testing only).
    - secrets_supports / shares_supports: public support points.
    - M_enc: encoding matrix for the *input* degree `indegree` (used to build shares_n).
    - M_dec: decoding matrix for secrets from n shares (must match supports/params).
    - V: Vandermonde matrix over `shares_supports` with shape (n x n), where V[i,j] = x_i^j.
    - coefficients_reconstruction(...): helper returning (coeff_list, degree_max) from shares.
    
    Failure behavior
    ----------------
    On any mismatch, the helper `_fail(...)` prints:
      - A clear failure message and parameter tuple (n, k, indegree, outdegree),
      - Half-splitting params (ceil/floor of n/2 and d/2) for context,
      - Extra debug fields (e.g., expected vs got, degree_max, nonzero coefficient indices,
        a preview of shares).
    Then it raises AssertionError to stop the test.
    
    Returns
    -------
    True  if all checks pass.
    
    Raises
    ------
    AssertionError on a deterministic correctness failure.
    Propagates other exceptions after printing a short crash banner.
    """
    # NOTE: we keep the same signature for compatibility.
    to_int = lambda x: (x.to_integer() if hasattr(x, "to_integer") else int(x))

    def _fail(msg, **extra):
        print("\n[ASSERT FAIL] test_M_lambda")
        print("  " + msg)
        print("  ---- parameters ----")
        print(f"    n={num_shares}, k={num_secrets}, indegree={indegree}, outdegree={outdegree}")
        print(f"    floor_d_half={floor_d_half}, ceil_d_half={ceil_d_half}, floor_n_half={floor_n_half}, ceil_n_half={ceil_n_half}")
        for k, v in extra.items():
            print(f"  {k}: {v}")
        raise AssertionError(msg)

    try:
        # sample k secrets and (indegree+1-k) extra coeffs uniformly at random
        secrets_k_plus_shares_d_plus_one_minus_k = generate_random_field_elements(ff, support_point_range, indegree + 1)
        secrets_k = secrets_k_plus_shares_d_plus_one_minus_k[:num_secrets]
        shares_d_plus_one_minus_k = secrets_k_plus_shares_d_plus_one_minus_k[num_secrets: indegree + 1]

        # generate n shares using k secrets and the indegree+1-k shares
        shares_n = pss_encoding(secrets_k, shares_d_plus_one_minus_k, M_enc, num_shares, indegree, num_secrets)

        # (re)generate M_lambda (ignore the incoming one; keep behavior explicit)
        M_lambda = generate_M_lambda(ff, secrets_supports, shares_supports, num_shares, indegree, outdegree)

        # apply degree reduction in the coefficient domain, then map back to the shares domain
        shares_n_outdegree_coefficients = M_lambda * matrix(shares_n).transpose()

        # V_submatrix should be n x (outdegree+1) so that V_submatrix * coeffs gives n shares
        V_submatrix = V.submatrix(0, 0, -1, outdegree + 1)
        shares_n_outdegree = V_submatrix * shares_n_outdegree_coefficients

        shares_n_outdegree_list = [shares_n_outdegree[i][0] for i in range(num_shares)]

        # 1) verify secrets from all n shares
        rec_shares_n_outdegree = pss_decoding(shares_n_outdegree_list, M_dec)
        for i in range(num_secrets):
            got = rec_shares_n_outdegree[i][0].to_integer()
            exp = secrets_k[i].to_integer()
            if got != exp:
                _fail(
                    f"secret mismatch (full decode) at i={i}",
                    expected=exp,
                    got=got,
                )

        # 2) verify secrets from any outdegree+1 shares
        rec_shares_n_outdegree_list = pss_decoding_from_all_partial_shares_combinations(
            ff, shares_supports, secrets_supports,
            num_shares, num_secrets,
            shares_n_outdegree_list, outdegree + 1
        )
        for j, rec in enumerate(rec_shares_n_outdegree_list):
            for i in range(num_secrets):
                got = rec[i][0].to_integer()
                exp = secrets_k[i].to_integer()
                if got != exp:
                    _fail(
                        f"secret mismatch (partial decode) subset={j}, i={i}",
                        subset=j,
                        expected=exp,
                        got=got,
                    )

        # 3) verify output degree is within (0, outdegree]
        coefficient_n_list, degree_max = coefficients_reconstruction(
            ff, shares_n_outdegree_list, shares_supports, num_shares
        )
        if not (0 < degree_max <= outdegree):
            # best-effort extra debug: show nonzero coefficient indices and tail
            try:
                nz = [idx for idx, c in enumerate(coefficient_n_list) if c != 0]
            except Exception:
                nz = None
            _fail(
                "degree check failed: expected 0 < degree_max <= outdegree",
                degree_max=degree_max,
                nonzero_coeff_indices=(nz[:40] if isinstance(nz, list) else nz),
                shares_preview=[to_int(x) for x in shares_n_outdegree_list[:min(10, len(shares_n_outdegree_list))]],
            )

        return True

    except AssertionError:
        # already printed via _fail
        raise
    except Exception as e:
        print("\n[ERROR] test_M_lambda crashed with non-Assertion error")
        print(f"  type={type(e).__name__}, msg={e}")
        print("  ---- parameters ----")
        print(f"    n={num_shares}, k={num_secrets}, indegree={indegree}, outdegree={outdegree}")
        raise

In [27]:
def test_M_lambda_with_permutation(
    ff, support_point_range,
    secrets_supports, shares_supports,
    num_secrets, num_shares,
    indegree, outdegree,
    M_enc, V,
    ceil_n_half, floor_n_half, ceil_d_half, floor_d_half
):
    """
    Test correctness of degree-reduction map M_lambda when the *secret support points*
    are permuted between "input secret positions" and "output secret positions".

    What this validates
    -------------------
    1) If we encode k secrets at secrets_supports_in (using M_enc),
       then apply generate_M_lambda_with_permutation() that targets secrets_supports_out,
       and then re-share back to n shares (via V_submatrix),
       decoding using secrets_supports_out must recover the original secrets.

    2) The above must hold:
       - for full reconstruction using all n shares, and
       - for partial reconstruction from any subset of size (outdegree + 1)
         (Shamir/PSS threshold at outdegree).

    3) The reconstructed polynomial degree from the output shares must satisfy:
         0 < degree_max <= outdegree
       (i.e., degree reduction actually happened and is within bound).

    Notes
    -----
    - This is a randomized test: it shuffles secret support points and samples random secrets/coeffs.
    - On failures, _fail() prints extensive debug context for fast diagnosis.
    """
    
    # local helpers (no extra global functions)
    to_int = lambda x: (x.to_integer() if hasattr(x, "to_integer") else int(x))

    def _preview(xs, k=10):
        try:
            xs = list(xs)
            return [to_int(xs[i]) for i in range(min(k, len(xs)))]
        except Exception:
            return "<unavailable>"

    def _fail(where, msg, **kwargs):
        print("\n[ASSERT FAIL] test_M_lambda_with_permutation")
        print(f"  where: {where}")
        print(f"  msg:   {msg}")
        print("  ---- parameters ----")
        print(f"    n={num_shares}, k={num_secrets}, indegree={indegree}, outdegree={outdegree}")
        print(f"    ceil_n_half={ceil_n_half}, floor_n_half={floor_n_half}, ceil_d_half={ceil_d_half}, floor_d_half={floor_d_half}")

        # permutation info
        try:
            print("  ---- permutation ----")
            print("    secrets_supports_in  (first 10):", _preview(secrets_supports_in, 10))
            print("    secrets_supports_out (first 10):", _preview(secrets_supports_out, 10))
            print("    permutation_list (first 30):", permutation_list[:30], ("..." if len(permutation_list) > 30 else ""))
        except Exception:
            pass

        # secrets / shares previews
        try:
            print("  ---- secrets_k (ints) ----")
            print("   ", _preview(secrets_k, 10))
        except Exception:
            pass

        try:
            print("  ---- shares_n preview (first 10, ints) ----")
            print("   ", _preview(shares_n, 10))
        except Exception:
            pass

        try:
            print("  ---- shares_n_outdegree_list preview (first 10, ints) ----")
            print("   ", _preview(shares_n_outdegree_list, 10))
        except Exception:
            pass

        # decoded secrets previews
        try:
            if rec_shares_n_outdegree_permutation is not None:
                print("  ---- decoded secrets (full) ----")
                got_full = [to_int(rec_shares_n_outdegree_permutation[i][0]) for i in range(min(num_secrets, len(rec_shares_n_outdegree_permutation)))]
                exp_full = [to_int(secrets_k[i]) for i in range(min(num_secrets, len(secrets_k)))]
                print("    got:", got_full)
                print("    exp:", exp_full)
        except Exception:
            pass

        # degree/coefficients info
        try:
            if degree_max is not None:
                print(f"  ---- reconstructed degree_max={degree_max} ----")
            if coefficient_n_list is not None:
                nz = [idx for idx, c in enumerate(coefficient_n_list) if c != 0]
                print("  nonzero coefficient indices:", nz[:40], ("..." if len(nz) > 40 else ""))
        except Exception:
            pass

        # any extra fields
        if kwargs:
            print("  ---- extra ----")
            for k, v in kwargs.items():
                try:
                    print(f"    {k}: {v}")
                except Exception:
                    print(f"    {k}: <unprintable>")

        raise AssertionError(f"{where}: {msg}")

    # keep vars defined for debug even if we crash mid-way
    secrets_supports_in = None
    secrets_supports_out = None
    permutation_list = None
    secrets_k_plus_shares_d_plus_one_minus_k = None
    secrets_k = None
    shares_d_plus_one_minus_k = None
    shares_n = None
    M_lambda_permutation = None
    shares_n_outdegree_coefficients = None
    V_submatrix = None
    shares_n_outdegree = None
    shares_n_outdegree_list = None
    M_dec_permutation = None
    rec_shares_n_outdegree_permutation = None
    rec_shares_n_outdegree_list = None
    coefficient_n_list = None
    degree_max = None

    try:
        secrets_supports_in = secrets_supports
        secrets_supports_out = list(secrets_supports_in)
        random.shuffle(secrets_supports_out)
        permutation_list = sort_list_by_reference(secrets_supports_in, secrets_supports_out)

        # sample k secrets and d+1-k shares uniformly at random
        secrets_k_plus_shares_d_plus_one_minus_k = generate_random_field_elements(
            ff, support_point_range, indegree + 1
        )
        secrets_k = secrets_k_plus_shares_d_plus_one_minus_k[:num_secrets]
        shares_d_plus_one_minus_k = secrets_k_plus_shares_d_plus_one_minus_k[num_secrets: indegree + 1]

        # encode
        shares_n = pss_encoding(
            secrets_k, shares_d_plus_one_minus_k, M_enc, num_shares, indegree, num_secrets
        )

        # degree reduction with permuted secrets supports
        M_lambda_permutation = generate_M_lambda_with_permutation(
            ff,
            secrets_supports_in, secrets_supports_out,
            shares_supports,
            num_shares,
            indegree, outdegree
        )

        shares_n_outdegree_coefficients = M_lambda_permutation * matrix(shares_n).transpose()
        V_submatrix = V.submatrix(0, 0, -1, outdegree + 1)
        shares_n_outdegree = V_submatrix * shares_n_outdegree_coefficients

        shares_n_outdegree_list = [shares_n_outdegree[i][0] for i in range(num_shares)]

        # decoder for permuted secret positions
        M_dec_permutation = generate_M_dec(ff, secrets_supports_out, shares_supports, num_shares, num_secrets)

        # 1) full decode secrets check
        rec_shares_n_outdegree_permutation = pss_decoding(shares_n_outdegree_list, M_dec_permutation)
        for i in range(num_secrets):
            got = rec_shares_n_outdegree_permutation[i][0].to_integer()
            exp = secrets_k[i].to_integer()
            if got != exp:
                _fail(
                    "full_decode_secret_check",
                    f"secret mismatch at i={i}",
                    i=i, expected=exp, got=got
                )

        # 2) partial decode check (any outdegree+1 shares)
        rec_shares_n_outdegree_list = pss_decoding_from_all_partial_shares_combinations(
            ff, shares_supports, secrets_supports_out,
            num_shares, num_secrets,
            shares_n_outdegree_list, outdegree + 1
        )
        for j, rec in enumerate(rec_shares_n_outdegree_list):
            for i in range(num_secrets):
                got = rec[i][0].to_integer()
                exp = secrets_k[i].to_integer()
                if got != exp:
                    _fail(
                        "partial_decode_secret_check",
                        f"secret mismatch at subset={j}, i={i}",
                        subset=j, i=i, expected=exp, got=got
                    )

        # 3) degree check
        coefficient_n_list, degree_max = coefficients_reconstruction(
            ff, shares_n_outdegree_list, shares_supports, num_shares
        )
        if not (0 < degree_max <= outdegree):
            try:
                nz = [idx for idx, c in enumerate(coefficient_n_list) if c != 0]
            except Exception:
                nz = None
            _fail(
                "degree_check",
                f"degree_max out of range: degree_max={degree_max}, expected 1..{outdegree}",
                degree_max=degree_max,
                outdegree=outdegree,
                nonzero_coeff_indices=(nz[:40] if isinstance(nz, list) else nz),
            )

        return True

    except AssertionError:
        # already printed by _fail
        raise
    except Exception as e:
        print("\n[ERROR] test_M_lambda_with_permutation crashed (non-Assertion)")
        print(f"  type={type(e).__name__}, msg={e}")
        print("  ---- parameters ----")
        print(f"    n={num_shares}, k={num_secrets}, indegree={indegree}, outdegree={outdegree}")
        raise

In [28]:
def test_split_red_degree_reduction(
    ff, support_point_range,
    secrets_supports, shares_supports,
    num_secrets, num_shares, degree,
    M_enc, M_dec,
    lambda_hat_matrix,
    ceil_n_half, floor_n_half,
    ceil_d_half, floor_d_half,
    degree_optzenc,
    verbose=False,
):
    """
    Test SplitRed (Algorithm 7) for *degree reduction correctness* and *secret preservation*.

    What this test is checking
    --------------------------
    Given a valid PSS sharing of k secrets using n shares with (input) degree bound d:

      1) SplitRed outputs two share vectors (f_prime, f_double_prime).
      2) Each *individual* output is expected to have degree > floor(d/2)
         (i.e., they are not already reduced on their own; they still contain
         higher-degree components due to the way SplitRed masks/re-randomizes).
      3) The *sum* f_prime + f_double_prime is expected to be a sharing whose
         underlying polynomial degree is <= floor(d/2) (degree reduction effect).
      4) Decoding f_prime + f_double_prime must recover the original secrets:
         - from all n shares (full reconstruction), and
         - from any (d+1) shares (threshold property consistent with the setup).

    Inputs/assumptions
    ------------------
    - ff: finite field
    - secrets_supports / shares_supports: public support points
    - M_enc / M_dec: encoding/decoding matrices consistent with (n, d, k) and supports
    - lambda_hat_matrix: combined degree-reduction + error-propagation map used by SplitRed
    - degree_optzenc: degree used for the optZEnc calls inside SplitRed (often floor(d/2))

    Returns
    -------
    True on success; raises AssertionError on failure with detailed diagnostics.
    """
    
    def _fail(msg, **kwargs):
        print("\n[FAIL] test_split_red_degree_reduction")
        print("  " + msg)
        print("  ---- parameters ----")
        print(f"    n = {num_shares}, d = {degree}, k = {num_secrets}")
        print(f"    floor(d/2) = {floor_d_half}, ceil(d/2) = {ceil_d_half}")
        print(f"    degree_optzenc = {degree_optzenc}")
        for k, v in kwargs.items():
            print(f"    {k} = {v}")
        raise AssertionError(msg)

    # ============================================================
    # 1. sample secrets and shares
    # ============================================================
    secrets_k_plus_shares_d_plus_one_minus_k = generate_random_field_elements(
        ff, support_point_range, degree + 1
    )
    secrets_k = secrets_k_plus_shares_d_plus_one_minus_k[:num_secrets]
    shares_d_plus_one_minus_k = secrets_k_plus_shares_d_plus_one_minus_k[num_secrets:]

    # encode
    shares_n = pss_encoding(
        secrets_k, shares_d_plus_one_minus_k,
        M_enc, num_shares, degree, num_secrets
    )

    # ============================================================
    # 2. SplitRed
    # ============================================================
    f_prime, f_double_prime = split_red(
        ff, support_point_range,
        shares_n,
        shares_supports, secrets_supports,
        num_shares, degree, num_secrets,
        ceil_n_half, floor_n_half,
        degree_optzenc,
        lambda_hat_matrix
    )

    # ============================================================
    # 3. check degrees of f' and f''
    # ============================================================
    _, deg_f_prime = coefficients_reconstruction(
        ff, f_prime, shares_supports, num_shares
    )
    _, deg_f_double_prime = coefficients_reconstruction(
        ff, f_double_prime, shares_supports, num_shares
    )

    if not (deg_f_prime > floor_d_half):
        _fail(
            "degree(f_prime) <= floor(d/2)",
            degree_f_prime=deg_f_prime,
        )

    if not (deg_f_double_prime > floor_d_half):
        _fail(
            "degree(f_double_prime) <= floor(d/2)",
            degree_f_double_prime=deg_f_double_prime,
        )

    # ============================================================
    # 4. add f' + f''
    # ============================================================
    f_sum = sw_add(f_prime, f_double_prime)

    # ============================================================
    # 5. secret correctness (all n shares)
    # ============================================================
    rec_secrets = pss_decoding(f_sum, M_dec)

    for i in range(num_secrets):
        if rec_secrets[i][0].to_integer() != secrets_k[i].to_integer():
            _fail(
                f"secret mismatch at index {i} (full reconstruction)",
                expected=secrets_k[i].to_integer(),
                got=rec_secrets[i][0].to_integer(),
            )

    # ============================================================
    # 6. secret correctness (any d+1 shares)
    # ============================================================
    rec_secrets_list = pss_decoding_from_all_partial_shares_combinations(
        ff,
        shares_supports, secrets_supports,
        num_shares, num_secrets,
        f_sum,
        degree + 1
    )

    for j, rec in enumerate(rec_secrets_list):
        for i in range(num_secrets):
            if rec[i][0].to_integer() != secrets_k[i].to_integer():
                _fail(
                    f"secret mismatch at subset {j}, index {i}",
                    subset=j,
                    expected=secrets_k[i].to_integer(),
                    got=rec[i][0].to_integer(),
                )

    # ============================================================
    # 7. final degree check
    # ============================================================
    _, degree_max = coefficients_reconstruction(
        ff, f_sum, shares_supports, num_shares
    )

    if not (0 < degree_max <= floor_d_half):
        _fail(
            "degree(f_prime + f_double_prime) out of range",
            degree_sum=degree_max,
        )

    if verbose:
        print(
            f"[OK] SplitRed degree reduction passed "
            f"(n={num_shares}, d={degree}, k={num_secrets}, "
            f"deg_sum={degree_max})"
        )

    return True

In [29]:
def test_split_red_error_propagation(
    ff, support_point_range, secrets_supports, shares_supports, num_secrets,
    num_shares, degree, M_enc, lambda_hat_matrix,
    ceil_n_half, floor_n_half, ceil_d_half, floor_d_half, degree_optzenc
):
    """
    Test SplitRed's fault amplification / error-propagation behavior.
    
    This test checks two complementary properties of SplitRed (Algorithm 7 / λ̂-based
    construction):
    
      (A) No-fault behavior (degree reduction):
          Starting from a valid sharing of degree <= d, SplitRed outputs two sharings
          f' and f'' whose sum f'+f'' has degree <= floor(d/2).
    
      (B) Fault behavior (degree amplification / error propagation):
          After injecting the maximum number of faults e = n - d - 1 into the share
          vector (i.e., corrupting e share positions arbitrarily), the SplitRed output
          sum should reflect the presence of high-degree terms: specifically, its
          reconstructed polynomial degree should become > d. This indicates that faults
          were propagated into the high-degree part as intended.
    
    Additionally, the test checks consistency: the reconstructed degree of the faulty
    input shares must match the reconstructed degree of the faulty SplitRed output sum.
    (This is a structural sanity check: SplitRed should not "invent" or "erase" degree
    beyond what the λ̂ + masking terms imply.)
    
    Returns
    -------
    True
        If all checks pass.
    
    Raises
    ------
    AssertionError
        If any degree condition fails.
    """

    # ===== Sample secrets and shares =====
    secrets_k_plus_shares_d_plus_one_minus_k = generate_random_field_elements(
        ff, support_point_range, degree + 1
    )
    secrets_k = secrets_k_plus_shares_d_plus_one_minus_k[:num_secrets]
    shares_d_plus_one_minus_k = secrets_k_plus_shares_d_plus_one_minus_k[num_secrets: degree + 1]

    shares_n = pss_encoding(
        secrets_k, shares_d_plus_one_minus_k, M_enc,
        num_shares, degree, num_secrets
    )

    # ===== Inject faults =====
    faulty_shares_n = list(shares_n)
    num_faults = num_shares - degree - 1
    faulty_share_index_list = np.random.choice(num_shares, num_faults, replace=False)

    for idx in faulty_share_index_list:
        faulty_share_poly = int_to_poly(
            ff, np.random.choice(support_point_range, 1, replace=False)[0]
        )
        faulty_shares_n[idx] = faulty_share_poly

    # ===== Apply SplitRed =====
    f_prime, f_double_prime = split_red(
        ff, support_point_range, shares_n,
        shares_supports, secrets_supports,
        num_shares, degree, num_secrets,
        ceil_n_half, floor_n_half,
        degree_optzenc, lambda_hat_matrix
    )

    faulty_f_prime, faulty_f_double_prime = split_red(
        ff, support_point_range, faulty_shares_n,
        shares_supports, secrets_supports,
        num_shares, degree, num_secrets,
        ceil_n_half, floor_n_half,
        degree_optzenc, lambda_hat_matrix
    )

    # ===== Degree after SplitRed (no fault) =====
    f_sum = sw_add(f_prime, f_double_prime)
    _, degree_max = coefficients_reconstruction(
        ff, f_sum, shares_supports, num_shares
    )

    # ===== Degree after SplitRed (faulty) =====
    faulty_f_sum = sw_add(faulty_f_prime, faulty_f_double_prime)
    _, degree_max_faulty_splitred = coefficients_reconstruction(
        ff, faulty_f_sum, shares_supports, num_shares
    )

    # ===== Degree of faulty inputs =====
    _, degree_max_faulty_inputs = coefficients_reconstruction(
        ff, faulty_shares_n, shares_supports, num_shares
    )

    # ===== Assertions with diagnostics =====
    if not (0 < degree_max <= floor_d_half):
        print("\n[ERROR] SplitRed error-propagation (no fault) degree violation")
        print(f"  degree_max           = {degree_max}")
        print(f"  floor_d_half         = {floor_d_half}")
        print(f"  params               = (n={num_shares}, d={degree}, k={num_secrets})")
        print(f"  degree_optzenc       = {degree_optzenc}")
        raise AssertionError("Degree after SplitRed exceeds floor(d/2).")

    if num_shares - 1 > degree:
        if not (degree_max_faulty_splitred > degree):
            print("\n[ERROR] Fault did NOT increase degree as expected")
            print(f"  degree_max_faulty    = {degree_max_faulty_splitred}")
            print(f"  original degree d    = {degree}")
            print(f"  faulty indices       = {sorted(faulty_share_index_list)}")
            print(f"  params               = (n={num_shares}, d={degree}, k={num_secrets})")
            print(f"  degree_optzenc       = {degree_optzenc}")
            raise AssertionError("Faulty SplitRed did not increase degree.")

    if degree_max_faulty_inputs != degree_max_faulty_splitred:
        print("\n[ERROR] Degree mismatch between faulty input and faulty SplitRed output")
        print(f"  degree_faulty_input  = {degree_max_faulty_inputs}")
        print(f"  degree_faulty_output = {degree_max_faulty_splitred}")
        print(f"  faulty indices       = {sorted(faulty_share_index_list)}")
        print(f"  params               = (n={num_shares}, d={degree}, k={num_secrets})")
        print(f"  degree_optzenc       = {degree_optzenc}")
        raise AssertionError("Fault degree mismatch before/after SplitRed.")

    return True

In [30]:
def test_binary_mult_LaOla(
    ff, support_point_range, secrets_supports, shares_supports,
    num_secrets, num_shares, degree,
    M_enc, M_dec, lambda_hat_matrix,
    ceil_n_half, floor_n_half, ceil_d_half, floor_d_half
):
    """
    Correctness test for LaOla-style masked multiplication in the *packed secret sharing*
    setting, using the SplitRed gadget + the local `binary_mult` composition rule.

    High-level goal
    ---------------
    This test checks that performing a masked multiplication on two shared inputs A and B:

        1) preserves the embedded secrets correctly (product of secrets),
        2) allows reconstruction from:
           - all n shares (full decode), and
           - any (degree + 1) shares (threshold decode),
        3) produces an output sharing whose reconstructed polynomial degree remains within
           the expected bound (0 < degree_max <= degree).

    What is being tested
    --------------------
    Pipeline under test:

      - Sample random inputs A and B (k secrets + (d+1-k) random coefficients each).
      - Encode them into n shares via `pss_encoding`.
      - Apply SplitRed to each sharing:
          * For A:  SplitRed(..., low_degree_optzenc=floor_d_half)
          * For B:  SplitRed(..., low_degree_optzenc=ceil_d_half)
        producing (f_prime, f_double_prime) for each input.
      - Compute the masked product share vector using:
            mult_result_shares_n = binary_mult(f'_A, f''_A, f'_B, f''_B)

    The correctness criteria are then verified by decoding and degree reconstruction.

    Parameters
    ----------
    ff : Sage finite field
        The finite field over which all operations are performed (e.g., GF(2^8)).
    support_point_range : iterable of int
        Candidate range of integers used to sample random field elements (test-only RNG).
    secrets_supports, shares_supports : list
        Public support points for secrets and shares (as field elements).
    num_secrets : int
        Number of packed secrets k.
    num_shares : int
        Number of shares n.
    degree : int
        Degree bound d for the sharing polynomial.
    M_enc : matrix
        Encoding matrix used by PSS to map (secrets || randomness) -> shares.
    M_dec : matrix
        Decoding matrix used to recover k secrets from n shares.
    lambda_hat_matrix : matrix
        Combined map used in SplitRed (degree reduction + error propagation).
    ceil_n_half, floor_n_half, ceil_d_half, floor_d_half : int
        Precomputed halves used in SplitRed (partitioning + degree targets).

    Returns
    -------
    True
        If all assertions pass.

    Raises
    ------
    AssertionError
        If any correctness condition fails. On failure, the function prints a detailed
        debug dump (parameters, secrets, share previews, SplitRed outputs, decoded secrets,
        reconstructed degree / nonzero coefficient positions) to help isolate issues.
    """
    
    # local helper only (not a new global function)
    to_int = lambda x: (x.to_integer() if hasattr(x, "to_integer") else int(x))

    # placeholders for debug dump
    secrets_k_A = secrets_k_B = None
    shares_n_A = shares_n_B = None
    f_prime_A = f_double_prime_A = None
    f_prime_B = f_double_prime_B = None
    mult_result_shares_n = None
    rec_mult_secrets = None
    rec_mult_secrets_list = None
    coefficient_n_list = None
    degree_max = None

    try:
        # ============================================================
        # generate shares_A
        # ============================================================
        secrets_k_plus_shares_d_plus_one_minus_k_A = generate_random_field_elements(
            ff, support_point_range, degree + 1
        )
        secrets_k_A = secrets_k_plus_shares_d_plus_one_minus_k_A[:num_secrets]
        shares_d_plus_one_minus_k_A = secrets_k_plus_shares_d_plus_one_minus_k_A[num_secrets: degree + 1]

        shares_n_A = pss_encoding(
            secrets_k_A, shares_d_plus_one_minus_k_A,
            M_enc, num_shares, degree, num_secrets
        )

        f_prime_A, f_double_prime_A = split_red(
            ff, support_point_range, shares_n_A,
            shares_supports, secrets_supports,
            num_shares, degree, num_secrets,
            ceil_n_half, floor_n_half,
            floor_d_half,  # degree_optzenc for A
            lambda_hat_matrix
        )

        # ============================================================
        # generate shares_B
        # ============================================================
        secrets_k_plus_shares_d_plus_one_minus_k_B = generate_random_field_elements(
            ff, support_point_range, degree + 1
        )
        secrets_k_B = secrets_k_plus_shares_d_plus_one_minus_k_B[:num_secrets]
        shares_d_plus_one_minus_k_B = secrets_k_plus_shares_d_plus_one_minus_k_B[num_secrets: degree + 1]

        shares_n_B = pss_encoding(
            secrets_k_B, shares_d_plus_one_minus_k_B,
            M_enc, num_shares, degree, num_secrets
        )

        f_prime_B, f_double_prime_B = split_red(
            ff, support_point_range, shares_n_B,
            shares_supports, secrets_supports,
            num_shares, degree, num_secrets,
            ceil_n_half, floor_n_half,
            ceil_d_half,  # degree_optzenc for B
            lambda_hat_matrix
        )

        # ============================================================
        # multiplication
        # ============================================================
        mult_result_shares_n = binary_mult(
            f_prime_A, f_double_prime_A,
            f_prime_B, f_double_prime_B
        )

        # ============================================================
        # verify secrets: full decode
        # ============================================================
        rec_mult_secrets = pss_decoding(mult_result_shares_n, M_dec)
        for i in range(num_secrets):
            got = rec_mult_secrets[i][0].to_integer()
            exp = (secrets_k_A[i] * secrets_k_B[i]).to_integer()
            assert got == exp, (
                f"[full decode] secret mismatch at i={i}: got={got}, expected={exp}"
            )

        # ============================================================
        # verify secrets: any (degree+1) shares
        # ============================================================
        rec_mult_secrets_list = pss_decoding_from_all_partial_shares_combinations(
            ff, shares_supports, secrets_supports,
            num_shares, num_secrets,
            mult_result_shares_n, degree + 1
        )
        for j, rec in enumerate(rec_mult_secrets_list):
            for i in range(num_secrets):
                got = rec[i][0].to_integer()
                exp = (secrets_k_A[i] * secrets_k_B[i]).to_integer()
                assert got == exp, (
                    f"[partial decode] secret mismatch at subset={j}, i={i}: got={got}, expected={exp}"
                )

        # ============================================================
        # verify degree
        # ============================================================
        coefficient_n_list, degree_max = coefficients_reconstruction(
            ff, mult_result_shares_n, shares_supports, num_shares
        )
        assert 0 < degree_max <= degree, (
            f"[degree] out of range: degree_max={degree_max}, expected 1..{degree}"
        )

        return True

    except AssertionError as e:
        # =========================
        # Detailed debug dump
        # =========================
        print("\n[ASSERT FAIL] test_binary_mult_LaOla")
        print("  ---- parameters ----")
        print(f"    n={num_shares}, d={degree}, k={num_secrets}")
        print(f"    ceil_n_half={ceil_n_half}, floor_n_half={floor_n_half}, ceil_d_half={ceil_d_half}, floor_d_half={floor_d_half}")

        # secrets
        try:
            print("  ---- secrets A/B (ints) ----")
            if secrets_k_A is not None:
                print("    secrets_k_A:", [to_int(x) for x in secrets_k_A])
            if secrets_k_B is not None:
                print("    secrets_k_B:", [to_int(x) for x in secrets_k_B])
        except Exception:
            pass

        # shares preview
        try:
            print("  ---- shares preview (first 10, ints) ----")
            if shares_n_A is not None:
                print("    shares_n_A:", [to_int(x) for x in shares_n_A[:min(10, len(shares_n_A))]])
            if shares_n_B is not None:
                print("    shares_n_B:", [to_int(x) for x in shares_n_B[:min(10, len(shares_n_B))]])
        except Exception:
            pass

        # split_red outputs preview
        try:
            print("  ---- split_red outputs preview (first 10, ints) ----")
            if f_prime_A is not None:
                print("    f_prime_A:", [to_int(x) for x in f_prime_A[:min(10, len(f_prime_A))]])
            if f_double_prime_A is not None:
                print("    f_double_prime_A:", [to_int(x) for x in f_double_prime_A[:min(10, len(f_double_prime_A))]])
            if f_prime_B is not None:
                print("    f_prime_B:", [to_int(x) for x in f_prime_B[:min(10, len(f_prime_B))]])
            if f_double_prime_B is not None:
                print("    f_double_prime_B:", [to_int(x) for x in f_double_prime_B[:min(10, len(f_double_prime_B))]])
        except Exception:
            pass

        # multiplication output preview
        try:
            if mult_result_shares_n is not None:
                print("  ---- mult_result_shares_n preview (first 10, ints) ----")
                print("   ", [to_int(x) for x in mult_result_shares_n[:min(10, len(mult_result_shares_n))]])
        except Exception:
            pass

        # decoded secrets preview
        try:
            if rec_mult_secrets is not None:
                print("  ---- decoded secrets (full) ----")
                print("   ", [to_int(rec_mult_secrets[i][0]) for i in range(min(num_secrets, len(rec_mult_secrets)))])
        except Exception:
            pass

        # degree / coefficients info
        try:
            if degree_max is not None:
                print(f"  ---- reconstructed degree_max={degree_max} ----")
            if coefficient_n_list is not None:
                nz = [idx for idx, c in enumerate(coefficient_n_list) if c != 0]
                print("  nonzero coefficient indices:", nz[:40], ("..." if len(nz) > 40 else ""))
        except Exception:
            pass

        print("  AssertionError:", str(e) if str(e) else "(no message)")
        raise

In [31]:
def test_square_LaOla(
    ff, support_point_range,
    secrets_supports, shares_supports,
    num_secrets, num_shares, degree,
    M_enc, M_dec, A_tilde, lambda_hat_matrix,
    ceil_n_half, floor_n_half, ceil_d_half, floor_d_half
):
    """
    Test: masked squaring via LaOla-based multiplication.
    
    This routine validates that `poly_masked_square(...)` computes the *share-domain*
    square of packed secrets correctly and keeps the resulting sharing within the
    intended degree bound.
    
    What is being tested
    --------------------
    1) Share generation:
       - Sample a random (d+1)-tuple of field elements.
       - Interpret the first k elements as packed secrets.
       - Encode them into n shares using PSS/Shamir-style encoding (`pss_encoding`).
    
    2) Gadget sanity (SplitRed):
       - Run `split_red` on the input sharing to ensure the environment/gadget
         machinery is exercised (and to catch obvious parameter mismatches early).
       - Note: SplitRed output is not directly asserted here; the main target is
         `poly_masked_square`.
    
    3) Square correctness:
       - Compute `square_result_shares_n = poly_masked_square(...)`.
       - Decode from:
           (a) all n shares, and
           (b) every subset of size (degree+1),
         and verify that each recovered secret equals (secret^2) in GF(2^m).
    
    4) Degree bound:
       - Reconstruct polynomial coefficients from all n shares and confirm the
         reconstructed degree lies in the expected range (1..degree).
    
    Expected behavior
    -----------------
    - For each secret index i in [0..k-1]:
        Decode(square(shares))[i] == (Decode(shares)[i])^2
      both for full reconstruction and any (degree+1)-subset reconstruction.
    - The output sharing degree must not exceed `degree`.
    
    Failure diagnostics
    -------------------
    On assertion failure, the function prints:
    - parameter summary (n, d, k and split sizes),
    - sampled secrets (integer encodings),
    - previews of the input shares and squared output shares,
    - decoded secrets (full reconstruction),
    - reconstructed output degree and indices of non-zero coefficients.
    
    Returns
    -------
    True
        If all correctness and degree checks pass.
    
    Raises
    ------
    AssertionError
        If any correctness/degree condition is violated.
    """
    to_int = lambda x: (x.to_integer() if hasattr(x, "to_integer") else int(x))

    try:
        # ============================================================
        # 1. generate shares_A
        # ============================================================
        secrets_k_plus_shares_d_plus_one_minus_k_A = generate_random_field_elements(
            ff, support_point_range, degree + 1
        )
        secrets_k_A = secrets_k_plus_shares_d_plus_one_minus_k_A[:num_secrets]
        shares_d_plus_one_minus_k_A = secrets_k_plus_shares_d_plus_one_minus_k_A[num_secrets: degree + 1]

        shares_n_A = pss_encoding(
            secrets_k_A, shares_d_plus_one_minus_k_A,
            M_enc, num_shares, degree, num_secrets
        )

        f_prime_A, f_double_prime_A = split_red(
            ff, support_point_range,
            shares_n_A,
            shares_supports, secrets_supports,
            num_shares, degree, num_secrets,
            ceil_n_half, floor_n_half,
            floor_d_half,
            lambda_hat_matrix
        )

        # ============================================================
        # 2. square
        # ============================================================
        square_result_shares_n = poly_masked_square(
            ff, support_point_range,
            shares_n_A,
            secrets_supports, shares_supports,
            num_secrets, num_shares, degree,
            A_tilde, lambda_hat_matrix,
            ceil_n_half, floor_n_half, ceil_d_half, floor_d_half
        )

        # ============================================================
        # 3. secret correctness (all n shares)
        # ============================================================
        rec_mult_secrets = pss_decoding(square_result_shares_n, M_dec)
        for i in range(num_secrets):
            got = rec_mult_secrets[i][0].to_integer()
            exp = (secrets_k_A[i] ** 2).to_integer()
            assert got == exp, (
                f"[full decode] secret mismatch at i={i}: got={got}, expected={exp}"
            )

        # ============================================================
        # 4. secret correctness (any degree+1 shares)
        # ============================================================
        rec_mult_secrets_list = pss_decoding_from_all_partial_shares_combinations(
            ff,
            shares_supports, secrets_supports,
            num_shares, num_secrets,
            square_result_shares_n,
            degree + 1
        )
        for j, rec in enumerate(rec_mult_secrets_list):
            for i in range(num_secrets):
                got = rec[i][0].to_integer()
                exp = (secrets_k_A[i] ** 2).to_integer()
                assert got == exp, (
                    f"[partial decode] subset={j}, i={i}: got={got}, expected={exp}"
                )

        # ============================================================
        # 5. degree check
        # ============================================================
        coefficient_n_list, degree_max = coefficients_reconstruction(
            ff, square_result_shares_n,
            shares_supports, num_shares
        )
        assert 0 < degree_max <= degree, (
            f"[degree] out of range: degree_max={degree_max}, expected 1..{degree}"
        )

        return True

    except AssertionError as e:
        # ============================================================
        # Detailed debug dump
        # ============================================================
        print("\n[ASSERT FAIL] test_square_LaOla")
        print("  ---- parameters ----")
        print(f"    n={num_shares}, d={degree}, k={num_secrets}")
        print(f"    ceil_n_half={ceil_n_half}, floor_n_half={floor_n_half}")
        print(f"    ceil_d_half={ceil_d_half}, floor_d_half={floor_d_half}")

        # secrets
        try:
            print("  ---- secrets_k_A (ints) ----")
            print("   ", [to_int(x) for x in secrets_k_A])
        except Exception:
            pass

        # shares preview
        try:
            print("  ---- shares_n_A preview (first 10, ints) ----")
            print("   ", [to_int(x) for x in shares_n_A[:min(10, len(shares_n_A))]])
        except Exception:
            pass

        # square result preview
        try:
            print("  ---- square_result_shares_n preview (first 10, ints) ----")
            print("   ", [to_int(x) for x in square_result_shares_n[:min(10, len(square_result_shares_n))]])
        except Exception:
            pass

        # decoded secrets preview
        try:
            print("  ---- decoded secrets (full) ----")
            print("   ", [to_int(rec_mult_secrets[i][0]) for i in range(min(num_secrets, len(rec_mult_secrets)))])
        except Exception:
            pass

        # degree / coefficients info
        try:
            print(f"  ---- reconstructed degree_max={degree_max} ----")
            nz = [idx for idx, c in enumerate(coefficient_n_list) if c != 0]
            print("  nonzero coefficient indices:", nz[:40], ("..." if len(nz) > 40 else ""))
        except Exception:
            pass

        print("  AssertionError:", str(e) if str(e) else "(no message)")
        raise

In [32]:
def test_poly_masked_sbox(
    ff, support_point_range, shares_supports, secrets_supports,
    num_shares, degree, num_secrets,
    ceil_n_half, floor_n_half, ceil_d_half, floor_d_half,
    M_enc, M_dec, A_tilde, lambda_hat_matrix
):
    """
    Test correctness of the masked AES S-box implementation based on
    polynomial evaluation and LaOla-style masked arithmetic.

    This test validates that:
      1) The masked S-box computation produces correct secrets after full decoding.
      2) Correctness holds for *any* subset of (degree + 1) shares (threshold security).
      3) The output sharing respects the expected polynomial degree bound.

    The test directly corresponds to the masked S-box construction described in:
      - "All-You-Can-Compute: Packed Secret Sharing for Combined Resilience"
      - CHES / TCHES artifact setting with SplitRed + optZEnc refresh.

    Parameters
    ----------
    ff : FiniteField
        Underlying finite field (typically GF(2^8) for AES).
    support_point_range : iterable
        Integer range used for random sampling of field elements.
    shares_supports, secrets_supports : list
        Public support points for shares and secrets.
    num_shares : int
        Number of shares n.
    degree : int
        Polynomial degree d of the input sharing.
    num_secrets : int
        Number of packed secrets k.
    ceil_n_half, floor_n_half : int
        Share split parameters used by SplitRed.
    ceil_d_half, floor_d_half : int
        Degree split parameters used by SplitRed.
    M_enc, M_dec : matrix
        Encoding and decoding matrices for PSS.
    A_tilde : matrix
        Zero-encoding matrix used for optZEnc / strong ZEnc.
    lambda_hat_matrix : matrix
        Combined degree-reduction + error-propagation matrix.

    Returns
    -------
    True
        If all correctness checks pass.

    Raises
    ------
    AssertionError
        If any correctness or threshold condition is violated.
    """
    
    to_int = lambda x: x.to_integer() if hasattr(x, "to_integer") else int(x)

    try:
        # ============================================================
        # 1. Sample secrets and encode
        # ============================================================
        secrets_k_plus_shares_d_plus_one_minus_k = generate_random_field_elements(
            ff, support_point_range, degree + 1
        )
        secrets_k = secrets_k_plus_shares_d_plus_one_minus_k[:num_secrets]
        shares_d_plus_one_minus_k = secrets_k_plus_shares_d_plus_one_minus_k[num_secrets: degree + 1]

        shares_n = pss_encoding(
            secrets_k, shares_d_plus_one_minus_k,
            M_enc, num_shares, degree, num_secrets
        )

        # ============================================================
        # 2. Masked S-box computation
        # ============================================================
        sbox_masked_result = poly_masked_sbox(
            ff, support_point_range,
            shares_n, shares_supports, secrets_supports,
            num_shares, degree, num_secrets,
            ceil_n_half, floor_n_half, ceil_d_half, floor_d_half,
            A_tilde, lambda_hat_matrix
        )

        # ============================================================
        # 3. Full decoding check
        # ============================================================
        rec_sbox_masked_result = pss_decoding(sbox_masked_result, M_dec)

        for i in range(num_secrets):
            got = rec_sbox_masked_result[i][0].to_integer()
            exp = sbox(ff, secrets_k[i]).to_integer()
            if got != exp:
                print("\n[ASSERT FAIL] test_poly_masked_sbox (full decode)")
                print(f"  index            = {i}")
                print(f"  secret            = {to_int(secrets_k[i])}")
                print(f"  expected sbox     = {exp}")
                print(f"  got               = {got}")
                raise AssertionError("S-box secret mismatch (full reconstruction)")

        # ============================================================
        # 4. Partial decoding check (any degree+1 shares)
        # ============================================================
        rec_sbox_masked_result_list = pss_decoding_from_all_partial_shares_combinations(
            ff, shares_supports, secrets_supports,
            num_shares, num_secrets,
            sbox_masked_result, degree + 1
        )

        for j, rec in enumerate(rec_sbox_masked_result_list):
            for i in range(num_secrets):
                got = rec[i][0].to_integer()
                exp = sbox(ff, secrets_k[i]).to_integer()
                if got != exp:
                    print("\n[ASSERT FAIL] test_poly_masked_sbox (partial decode)")
                    print(f"  subset index      = {j}")
                    print(f"  secret index      = {i}")
                    print(f"  secret            = {to_int(secrets_k[i])}")
                    print(f"  expected sbox     = {exp}")
                    print(f"  got               = {got}")
                    raise AssertionError("S-box secret mismatch (partial reconstruction)")

        # ============================================================
        # 5. Degree check
        # ============================================================
        coefficient_n_list, degree_max = coefficients_reconstruction(
            ff, sbox_masked_result, shares_supports, num_shares
        )

        if not (0 <= degree_max <= degree):
            nz = [i for i, c in enumerate(coefficient_n_list) if c != 0]
            print("\n[ASSERT FAIL] test_poly_masked_sbox (degree check)")
            print(f"  degree_max        = {degree_max}")
            print(f"  expected range    = [0, {degree}]")
            print(f"  nonzero coeff idx = {nz[:40]}{'...' if len(nz) > 40 else ''}")
            raise AssertionError("Degree of masked S-box output out of range")

        return True

    except AssertionError:
        # ============================================================
        # Global debug dump
        # ============================================================
        print("\n[DEBUG DUMP] test_poly_masked_sbox context")
        print("  ---- parameters ----")
        print(f"    n={num_shares}, d={degree}, k={num_secrets}")
        print(f"    floor_d_half={floor_d_half}, ceil_d_half={ceil_d_half}")
        print(f"    floor_n_half={floor_n_half}, ceil_n_half={ceil_n_half}")

        try:
            print("  ---- secrets (ints) ----")
            print("   ", [to_int(x) for x in secrets_k])
        except Exception:
            pass

        try:
            print("  ---- shares_n preview (first 10) ----")
            print("   ", [to_int(x) for x in shares_n[:min(10, len(shares_n))]])
        except Exception:
            pass

        try:
            print("  ---- sbox_masked_result preview (first 10) ----")
            print("   ", [to_int(x) for x in sbox_masked_result[:min(10, len(sbox_masked_result))]])
        except Exception:
            pass

        raise

In [33]:
def test_matrix_lambda_degree_reduction_shamir(ff, support_point_range, num_shares, indegree, outdegree):
    # local helpers (no extra global functions)
    to_int = lambda x: (x.to_integer() if hasattr(x, "to_integer") else int(x))

    def _preview_list(xs, k=10):
        try:
            xs = list(xs)
            return [to_int(xs[i]) for i in range(min(k, len(xs)))]
        except Exception:
            return "<unavailable>"

    # keep vars for debug even if we crash early
    secrets_plus_indegree_poly_coefficients_plus_n_supports = None
    secrets_plus_indegree_poly_coefficients = None
    n_supports = None
    V = V_inv = None
    shares_n_indegree = None
    lambda_degree_reduction = None
    shares_n_outdegree = None
    rec_secrets_outdegree = None

    try:
        # sample 1 secret, n support points, and indegree random polynomial coefficients
        secrets_plus_indegree_poly_coefficients_plus_n_supports = generate_random_field_elements(
            ff, support_point_range, num_shares + 1 + indegree
        )

        # split into coeffs + supports
        secrets_plus_indegree_poly_coefficients = secrets_plus_indegree_poly_coefficients_plus_n_supports[: 1 + indegree]
        n_supports = secrets_plus_indegree_poly_coefficients_plus_n_supports[1 + indegree :]

        # generate Vandermonde matrix V and inverse V_inv
        V, V_inv = generate_Vs(ff, n_supports, num_shares)

        # encode indegree polynomial into n shares
        shares_n_indegree = V[:, : indegree + 1] * (matrix(secrets_plus_indegree_poly_coefficients).transpose())

        # reduce the degree using lambda_degree_reduction
        lambda_degree_reduction = generate_lambda_degree_reduction_shamir(
            ff, V, V_inv, num_shares, indegree, outdegree
        )
        shares_n_outdegree = lambda_degree_reduction * shares_n_indegree

        # reconstruct coefficients from reduced shares
        rec_secrets_outdegree = V_inv * shares_n_outdegree

        for i in range(num_shares):
            got = rec_secrets_outdegree[i][0]
            if i < outdegree + 1:
                exp = secrets_plus_indegree_poly_coefficients[i]
                assert got.to_integer() == exp.to_integer(), (
                    f"[coeff match] i={i}: got={to_int(got)} expected={to_int(exp)} "
                    f"(should preserve coeffs 0..{outdegree})"
                )
            else:
                assert got.to_integer() == 0, (
                    f"[tail zero] i={i}: got={to_int(got)} expected=0 "
                    f"(coeffs {outdegree+1}..{num_shares-1} must be zero)"
                )

        return True

    except AssertionError as e:
        print("\n[ASSERT FAIL] test_matrix_lambda_degree_reduction_shamir")
        print("  ---- parameters ----")
        print(f"    n={num_shares}, indegree={indegree}, outdegree={outdegree}")
        try:
            print("  ---- supports (n_supports) preview ----")
            print("    n_supports (first 10):", _preview_list(n_supports, 10))
        except Exception:
            pass

        try:
            print("  ---- original coeffs preview ----")
            print("    coeffs (first 10):", _preview_list(secrets_plus_indegree_poly_coefficients, 10))
        except Exception:
            pass

        try:
            # show reconstructed coeffs (first 10)
            if rec_secrets_outdegree is not None:
                rec0 = [to_int(rec_secrets_outdegree[i][0]) for i in range(min(10, num_shares))]
                print("  ---- reconstructed coeffs preview (first 10) ----")
                print("   ", rec0)
        except Exception:
            pass

        try:
            # show share previews
            if shares_n_indegree is not None:
                indeg_sh = [to_int(shares_n_indegree[i][0]) for i in range(min(10, num_shares))]
                print("  ---- shares_n_indegree preview (first 10 shares) ----")
                print("   ", indeg_sh)
            if shares_n_outdegree is not None:
                outdeg_sh = [to_int(shares_n_outdegree[i][0]) for i in range(min(10, num_shares))]
                print("  ---- shares_n_outdegree preview (first 10 shares) ----")
                print("   ", outdeg_sh)
        except Exception:
            pass

        print("  AssertionError:", str(e) if str(e) else "(no message)")
        raise

    except Exception as e:
        print("\n[ERROR] test_matrix_lambda_degree_reduction_shamir crashed (non-Assertion)")
        print(f"  type={type(e).__name__}, msg={e}")
        print("  ---- parameters ----")
        print(f"    n={num_shares}, indegree={indegree}, outdegree={outdegree}")
        raise


In [34]:
# ============================================================
# Test: Zero-Encodings R for A_hat (Shamir Case)
# ============================================================
#
# This test validates the correctness of the special zero-encodings R^{(k)}
# used in the construction of the A_hat matrix for the adapted SplitRed gadget
# (Shamir secret sharing case).
#
# Setting:
#   - Shamir secret sharing with n shares
#   - input polynomial degree = indegree = d
#   - target reduced degree = outdegree = floor(d/2)
#   - number of tolerable faults e = n - d - 1
#
# Construction under test:
#   For each k ∈ {0, …, e−1}, generate a zero-encoding R^{(k)} such that
#   its polynomial coefficients satisfy:
#
#       coeff_0              = random
#       coeff_{outdegree+1}  = random
#       ...
#       coeff_{n-1}          = random
#       all other coefficients = 0
#
#   Equivalently, R^{(k)} has:
#     - no secret embedded (zero-encoding),
#     - randomness only in the constant term and in the high-degree part
#       (degrees > outdegree),
#     - zero coefficients in degrees 1 … outdegree.
#
# Test procedure:
#   1) Randomly sample n distinct support points.
#   2) Construct the Vandermonde matrix V and its inverse V_inv.
#   3) Generate all e zero-encodings R^{(0)}, …, R^{(e−1)} using
#      generate_ZEnc_R_for_A_hat_shamir().
#   4) Reconstruct the polynomial coefficients of each R^{(k)} via V_inv.
#   5) Verify that:
#        - coefficient 0 matches the expected random value,
#        - coefficients (outdegree+1 … n−1) match the expected random values,
#        - coefficients (1 … outdegree) are exactly zero.
#
# This test ensures that the zero-encodings R used in A_hat:
#   - correctly isolate high-degree randomness,
#   - do not affect low-degree coefficients,
#   - are suitable for fault-triggered re-randomization in SplitRed.
#
# The test is deterministic modulo field randomness and should pass
# with probability 1.
# ============================================================


def test_generate_ZEnc_R_for_A_hat_shamir(ff, support_point_range, num_shares, indegree, outdegree):
    num_faults = num_shares-indegree-1
    
    # Generate randomness for the e zero-encodings, each needs n-outdegree-1 random values
    random_poly_list = generate_random_field_elements(ff, support_point_range, num_faults*(num_shares-outdegree))

    # Generate n support points for sharing
    n_supports = generate_random_field_elements(ff, support_point_range, num_shares)
    
    # generate Vandermonde matrix v, and inverse Vandermonde matrix
    V, V_inv = generate_Vs(ff, n_supports, num_shares)
      
    ZEnc_R_list = generate_ZEnc_R_for_A_hat_shamir(ff, V, num_shares, indegree, outdegree, random_poly_list)    

    # reconstruct the coefficients of 'e' zero-encodings and verify if only the 0, d/2+1, ..., n-1 coefficients are the previously generated random, the rest coefficients are zeros
    for k in range(num_shares-indegree-1):
        ZEnc_R_k_coefficients = V_inv * (matrix(ZEnc_R_list[k]).transpose())
        for j in range(num_shares):
            if j==0:
                assert (ZEnc_R_k_coefficients[j][0]).to_integer() == (random_poly_list[k*(num_shares-outdegree-1)+0]).to_integer()
            elif ((j>outdegree) & (j<num_shares)):
                assert (ZEnc_R_k_coefficients[j][0]).to_integer() == (random_poly_list[k*(num_shares-outdegree-1)+(j-outdegree)]).to_integer()
            else:
                assert (ZEnc_R_k_coefficients[j][0]).to_integer() == 0
    return True

In [35]:
def test_generate_A_hat_shamir(ff, support_point_range, num_shares, indegree, outdegree):
    num_faults = num_shares - indegree - 1

    # small local helpers (inside function; not extra global functions)
    to_int = lambda x: (x.to_integer() if hasattr(x, "to_integer") else int(x))

    def _preview_vec(v, k=10):
        try:
            v_list = list(v)
            return [to_int(v_list[i]) for i in range(min(k, len(v_list)))]
        except Exception:
            return "<unavailable>"

    def _preview_mat_col(M, col=0, k=10):
        try:
            # assume sage matrix-like indexing M[i][j]
            return [to_int(M[i][col]) for i in range(min(k, M.nrows()))]
        except Exception:
            return "<unavailable>"

    # keep some vars for debug printing even if we crash early
    secrets_plus_indegree_poly_coefficients = None
    n_supports = None
    V = V_inv = None
    matrix_A_hat_shamir = None
    shares_n_indegree = None
    shares_n_outdegree = None
    rec_secrets_outdegree = None
    faulty_shares_n = None
    faulty_share_index_list = None
    faulty_shares_n_after_coefficients_randomization = None
    rec_secrets_randomized_coefficients = None

    try:
        # sample 1 secret, n support points, and indegree random polynomial coefficients
        secrets_plus_indegree_poly_coefficients_plus_n_supports = generate_random_field_elements(
            ff, support_point_range, num_shares + 1 + indegree
        )

        # secret + coeffs (length 1+indegree), then n support points (length num_shares)
        secrets_plus_indegree_poly_coefficients = secrets_plus_indegree_poly_coefficients_plus_n_supports[: 1 + indegree]
        n_supports = secrets_plus_indegree_poly_coefficients_plus_n_supports[1 + indegree :]

        # Vandermonde and inverse
        V, V_inv = generate_Vs(ff, n_supports, num_shares)

        # randomness for e zero-encodings (as in your original code)
        random_poly_list = generate_random_field_elements(
            ff, support_point_range, num_faults * (num_shares - outdegree)
        )

        matrix_A_hat_shamir = generate_A_hat_shamir(
            ff, V, V_inv, num_shares, indegree, outdegree, random_poly_list
        )

        # shares of degree 'indegree'
        shares_n_indegree = V[:, : indegree + 1] * (matrix(secrets_plus_indegree_poly_coefficients).transpose())

        # ==========================================================================================
        # No faults: degree reduction check (coeff-domain check via V_inv)
        # ==========================================================================================
        shares_n_outdegree = matrix_A_hat_shamir * shares_n_indegree
        rec_secrets_outdegree = V_inv * shares_n_outdegree

        for i in range(num_shares):
            got = (rec_secrets_outdegree[i][0]).to_integer()
            if i < outdegree + 1:
                exp = (secrets_plus_indegree_poly_coefficients[i]).to_integer()
                assert got == exp, (
                    f"[no-fault] coeff mismatch at i={i}: expected={exp}, got={got} "
                    f"(indegree={indegree}, outdegree={outdegree}, n={num_shares})"
                )
            else:
                assert got == 0, (
                    f"[no-fault] high-degree coeff not zero at i={i}: got={got}, expected=0 "
                    f"(indegree={indegree}, outdegree={outdegree}, n={num_shares})"
                )

        # ==========================================================================================
        # Faults: coefficient randomization check (probabilistic)
        # ==========================================================================================
        faulty_shares_n = [shares_n_indegree[share_index][0] for share_index in range(num_shares)]
        faulty_share_index_list = np.random.choice(num_shares, num_faults, replace=False)

        for idx in faulty_share_index_list:
            faulty_share_poly = int_to_poly(ff, np.random.choice(support_point_range, 1, replace=False)[0])
            faulty_shares_n[idx] = faulty_share_poly

        faulty_shares_n_after_coefficients_randomization = matrix_A_hat_shamir * (matrix(faulty_shares_n).transpose())
        rec_secrets_randomized_coefficients = V_inv * faulty_shares_n_after_coefficients_randomization

        for i in range(num_shares):
            got = (rec_secrets_randomized_coefficients[i][0]).to_integer()

            # NOTE: these are "high probability" checks; still can fail rarely by chance.
            if i == 0:
                exp = (secrets_plus_indegree_poly_coefficients[i]).to_integer()
                assert got != exp, (
                    f"[fault] coeff(0) not randomized (rare-collision possible): got={got} equals original={exp}. "
                    f"faulty_indices={sorted(map(int, faulty_share_index_list))}"
                )
            elif (i > outdegree) and (i < indegree + 1):
                exp = (secrets_plus_indegree_poly_coefficients[i]).to_integer()
                assert got != exp, (
                    f"[fault] coeff({i}) not randomized (rare-collision possible): got={got} equals original={exp}. "
                    f"faulty_indices={sorted(map(int, faulty_share_index_list))}"
                )
            else:
                assert got != 0, (
                    f"[fault] coeff({i}) stayed zero (rare-collision possible): got=0. "
                    f"faulty_indices={sorted(map(int, faulty_share_index_list))}"
                )

        return True

    except AssertionError as e:
        # =========================
        # Detailed debug dump
        # =========================
        print("\n[ASSERT FAIL] test_generate_A_hat_shamir")
        print("  ---- parameters ----")
        print(f"    n={num_shares}, indegree={indegree}, outdegree={outdegree}, num_faults={num_faults}")

        # supports / coefficients
        try:
            print("  ---- support points (n_supports) preview ----")
            print("    n_supports (first 10):", _preview_vec(n_supports, 10))
        except Exception:
            pass

        try:
            print("  ---- secret+coeffs preview ----")
            print("    secrets_plus_indegree_poly_coefficients (as ints, first 10):",
                  _preview_vec(secrets_plus_indegree_poly_coefficients, 10))
        except Exception:
            pass

        # fault indices
        try:
            print("  ---- faults ----")
            print("    faulty_share_index_list:", sorted(map(int, faulty_share_index_list)) if faulty_share_index_list is not None else None)
        except Exception:
            pass

        # shares previews
        try:
            print("  ---- shares_n_indegree preview (first 10 shares ints) ----")
            if shares_n_indegree is not None:
                # shares_n_indegree is an (n x 1) matrix
                print("   ", _preview_mat_col(shares_n_indegree, 0, 10))
        except Exception:
            pass

        try:
            print("  ---- no-fault reconstructed coeffs preview (first 10) ----")
            if rec_secrets_outdegree is not None:
                print("   ", _preview_mat_col(rec_secrets_outdegree, 0, 10))
        except Exception:
            pass

        try:
            print("  ---- faulty reconstructed coeffs preview (first 10) ----")
            if rec_secrets_randomized_coefficients is not None:
                print("   ", _preview_mat_col(rec_secrets_randomized_coefficients, 0, 10))
        except Exception:
            pass

        # note about probabilistic asserts
        print("  ---- note ----")
        print("    Some assertions in the fault-randomization part are probabilistic;")
        print("    rare collisions can make them fail even if the construction is correct.")
        print("  AssertionError:", str(e) if str(e) else "(no message)")
        raise


## Comprehensive Correctness, Robustness, and Fault-Propagation Test Harness

This section implements a **systematic and executable test harness** for validating the
core correctness, robustness, and fault-related guarantees of the constructions
presented in *“All-You-Can-Compute: Packed Secret Sharing for Combined Resilience”*.

The tests in this harness collectively verify that the implemented primitives satisfy
their intended properties under the combined ((t, e))-adversary model, including:

* **Functional correctness**:
  Ensuring that all primitives (zero-encodings, degree reduction, SplitRed,
  LaOla multiplication, and the masked AES S-box) correctly preserve and reconstruct
  the embedded secrets.

* **Degree preservation and reduction**:
  Verifying that polynomial degrees remain within the specified bounds at every stage,
  and that degree-reduction gadgets reduce the degree exactly as required.

* **Threshold reconstruction guarantees**:
  Confirming that secrets can be reconstructed from *all* shares as well as from
  *any qualifying subset* of shares, consistent with the underlying Shamir/PSS model.

* **Fault propagation and amplification behavior**:
  Explicitly testing that injected faults lead to the appearance of high-degree
  polynomial components and that the corresponding mechanisms (e.g., λ̂ and Â)
  correctly randomize or amplify these components as intended.

The harness is designed as an **end-to-end validation framework** rather than a
collection of isolated unit tests. On failure, each test emits detailed diagnostic
information (shares, coefficients, degrees, and decoded secrets) to support
fast debugging and artifact evaluation.


In [36]:
def build_env(num_shares, degree, num_secrets, seed=None):
    # Ensure reproducibility across NumPy, Python, and Sage RNGs
    if seed is not None:
        seed = int(seed)          # enforce Python int
        np.random.seed(seed)
        random.seed(seed)
        set_random_seed(seed)     # Sage RNG

    # Finite field GF(2^8) for AES S-box
    ff.<a> = GF(2**8, name='a', modulus=x^8 + x^4 + x^3 + x + 1)

    indegree  = degree
    outdegree = floor(degree / 2)
    num_faults = num_shares - degree - 1

    floor_n_half = floor(num_shares / 2)
    ceil_n_half  = ceil(num_shares / 2)
    floor_d_half = floor(degree / 2)
    ceil_d_half  = ceil(degree / 2)

    # Support points
    support_point_range = range(1, 2**8 - 1)  # 1..254
    support_points_int = np.random.choice(support_point_range, 1 + degree + num_shares, replace=False)
    support_points_poly = intlist_to_polylist(ff, support_points_int)

    # Shamir supports: secrets_supports (degree+1), shares_supports (n)
    secrets_supports = support_points_poly[:degree + 1]
    shares_supports  = support_points_poly[degree + 1:]

    # Expensive matrices (compute once)
    V, V_inv = generate_Vs(ff, shares_supports, num_shares)

    M_enc = generate_M_enc(ff, secrets_supports, shares_supports, num_secrets, num_shares, degree)
    M_dec = generate_M_dec(ff, secrets_supports, shares_supports, num_shares, num_secrets)
    A_tilde = generate_A_tilde(ff, M_enc, num_secrets, degree)

    M_lambda = generate_M_lambda(ff, secrets_supports, shares_supports, num_shares, indegree, outdegree)

    error_propagation_matrix = generate_error_propagation_matrix(indegree, V, V_inv)
    lambda_hat_matrix = generate_lambda_hat_matrix(V, M_lambda, error_propagation_matrix, outdegree)

    return {
        "ff": ff,
        "support_point_range": support_point_range,
        "secrets_supports": secrets_supports,
        "shares_supports": shares_supports,
        "num_shares": num_shares,
        "degree": degree,
        "num_secrets": num_secrets,
        "indegree": indegree,
        "outdegree": outdegree,
        "num_faults": num_faults,
        "floor_n_half": floor_n_half,
        "ceil_n_half": ceil_n_half,
        "floor_d_half": floor_d_half,
        "ceil_d_half": ceil_d_half,
        "V": V,
        "V_inv": V_inv,
        "M_enc": M_enc,
        "M_dec": M_dec,
        "A_tilde": A_tilde,
        "M_lambda": M_lambda,
        "error_propagation_matrix": error_propagation_matrix,
        "lambda_hat_matrix": lambda_hat_matrix,
    }


# ===== Test Execution and Validation =====
def run_tests(env, quick=False):
    ff = env["ff"]
    support_point_range = env["support_point_range"]
    secrets_supports = env["secrets_supports"]
    shares_supports = env["shares_supports"]
    num_secrets = env["num_secrets"]
    num_shares  = env["num_shares"]
    degree      = env["degree"]
    indegree    = env["indegree"]
    outdegree   = env["outdegree"]

    M_enc = env["M_enc"]
    M_dec = env["M_dec"]
    V     = env["V"]
    V_inv = env["V_inv"]
    M_lambda = env["M_lambda"]
    A_tilde  = env["A_tilde"]
    lambda_hat_matrix = env["lambda_hat_matrix"]

    floor_n_half = env["floor_n_half"]
    ceil_n_half  = env["ceil_n_half"]
    floor_d_half = env["floor_d_half"]
    ceil_d_half  = env["ceil_d_half"]

    # 1) optZEnc: different randoffset
    for randoffset in range(degree + 1 - num_secrets):
        test_optZEnc_randoffset(
            ff, support_point_range, secrets_supports, shares_supports,
            num_secrets, num_shares, degree, randoffset, M_dec
        )
        if quick:
            break

    # 2) optZEnc: different degree_optzenc
    for degree_optzenc in range(num_secrets, num_shares):
        randoffset = 0
        # test_optZEnc_degree is assumed to generate its own randomness internally
        test_optZEnc_degree(
            ff, support_point_range, secrets_supports, shares_supports,
            num_secrets, num_shares, degree_optzenc, randoffset, M_dec
        )
        if quick:
            break

    # 3) optsZEnc
    test_optsZEnc(ff, support_point_range, secrets_supports, shares_supports,
                  num_secrets, num_shares, degree, M_dec, A_tilde)

    # 4) M_lambda + permutation
    test_M_lambda(ff, support_point_range, secrets_supports, shares_supports,
                  num_secrets, num_shares, indegree, outdegree, M_enc, M_dec,
                  V, ceil_n_half, floor_n_half, ceil_d_half, floor_d_half)

    if not quick:
        test_M_lambda_with_permutation(ff, support_point_range, secrets_supports, shares_supports,
                                       num_secrets, num_shares, indegree, outdegree, M_enc,
                                       V, ceil_n_half, floor_n_half, ceil_d_half, floor_d_half)

    # 5) SplitRed
    degree_optzenc = floor_d_half
    test_split_red_degree_reduction(ff, support_point_range, secrets_supports, shares_supports,
                                    num_secrets, num_shares, degree, M_enc, M_dec,
                                    lambda_hat_matrix,
                                    ceil_n_half, floor_n_half, ceil_d_half, floor_d_half, degree_optzenc)

    if not quick:
        test_split_red_error_propagation(ff, support_point_range, secrets_supports, shares_supports,
                                         num_secrets, num_shares, degree, M_enc,
                                         lambda_hat_matrix,
                                         ceil_n_half, floor_n_half, ceil_d_half, floor_d_half, degree_optzenc)

    # 6) LaOla mult / square / sbox
    test_binary_mult_LaOla(ff, support_point_range, secrets_supports, shares_supports,
                           num_secrets, num_shares, degree, M_enc, M_dec,
                           lambda_hat_matrix,
                           ceil_n_half, floor_n_half, ceil_d_half, floor_d_half)

    if not quick:
        test_square_LaOla(ff, support_point_range, secrets_supports, shares_supports,
                          num_secrets, num_shares, degree, M_enc, M_dec,
                          A_tilde, lambda_hat_matrix,
                          ceil_n_half, floor_n_half, ceil_d_half, floor_d_half)

        test_poly_masked_sbox(ff, support_point_range, shares_supports, secrets_supports,
                              num_shares, degree, num_secrets,
                              ceil_n_half, floor_n_half, ceil_d_half, floor_d_half,
                              M_enc, M_dec, A_tilde, lambda_hat_matrix)

    # 7) A_hat tests
    test_matrix_lambda_degree_reduction_shamir(ff, support_point_range, num_shares, indegree, outdegree)
    test_generate_ZEnc_R_for_A_hat_shamir(ff, support_point_range, num_shares, indegree, outdegree)

    if not quick:
        test_generate_A_hat_shamir(ff, support_point_range, num_shares, indegree, outdegree)

    return True


def test_environment(num_shares, degree, num_secrets, seed=None, quick=False, verbose=False):
    """
    Build the environment and run all tests.

    verbose=False ensures that Monte Carlo loops do not print per-iteration messages,
    allowing tqdm to update a single progress line.
    """
    env = build_env(num_shares, degree, num_secrets, seed=seed)
    run_tests(env, quick=quick)

    if verbose:
        print("All tests have passed.")

    return True


# ===== Monte Carlo Test Loop (Single Parameter Set) =====
def run_monte_carlo_tests(
    num_tests,
    num_shares,
    degree,
    num_secrets,
    *,
    base_seed=1,
    vary_seed=True,
    quick=False,
    stop_after_failures=None,
    verbose_fail=True,
):
    results = Counter()
    failure_cases = []

    for i in tqdm(range(num_tests), desc="Running tests", leave=True):
        seed = int((base_seed + i) if vary_seed else base_seed)

        try:
            test_environment(
                num_shares,
                degree,
                num_secrets,
                seed=seed,
                quick=quick,
                verbose=False,   # IMPORTANT: keep tqdm on a single line
            )
            results["pass"] += 1

        except AssertionError as e:
            results["fail"] += 1
            failure_cases.append((i, seed, repr(e)))
            if verbose_fail:
                tqdm.write(f"[FAIL] i={i}, seed={seed}, err={repr(e)}")

            if stop_after_failures is not None and results["fail"] >= stop_after_failures:
                tqdm.write(f"Stopped early after {results['fail']} failures.")
                break

        except Exception as e:
            results["error"] += 1
            failure_cases.append((i, seed, f"Unexpected: {type(e).__name__}: {e}"))
            if verbose_fail:
                tqdm.write(f"[ERROR] i={i}, seed={seed}, {type(e).__name__}: {e}")

            if stop_after_failures is not None and (results["fail"] + results["error"]) >= stop_after_failures:
                tqdm.write(f"Stopped early after {results['fail'] + results['error']} failures/errors.")
                break

    total = sum(results.values())
    print("\n===== Summary =====")
    print(f"Total runs: {total}")
    print(f"Pass:  {results['pass']}")
    print(f"Fail:  {results['fail']}")
    print(f"Error: {results['error']}")

    if failure_cases:
        print("\nFirst few failures/errors (index, seed, message):")
        for row in failure_cases[:5]:
            print("  ", row)

    return {
        "results": dict(results),
        "failure_cases": failure_cases,
    }

## Example Run: Algorithm Validation for a Fixed (n, d, k)

In [37]:
# --- Example run ---
num_tests   = 100
num_shares  = 10
degree      = 7
num_secrets = 3

assert_valid_params(num_shares, degree, num_secrets, strict=True)

report = run_monte_carlo_tests(
    num_tests,
    num_shares,
    degree,
    num_secrets,
    base_seed=1,
    vary_seed=True,
    quick=False,
    stop_after_failures=None,
    verbose_fail=True,
)

Running tests:   2%|████▎                                                                                                                                                                                                                    | 2/100 [00:00<00:38,  2.56it/s]


[ASSERT FAIL] test_generate_A_hat_shamir
  ---- parameters ----
    n=10, indegree=7, outdegree=3, num_faults=2
  ---- support points (n_supports) preview ----
    n_supports (first 10): [158, 101, 211, 191, 162, 46, 134, 45, 122, 103]
  ---- secret+coeffs preview ----
    secrets_plus_indegree_poly_coefficients (as ints, first 10): [10, 148, 8, 220, 183, 252, 130, 66]
  ---- faults ----
    faulty_share_index_list: [1, 4]
  ---- shares_n_indegree preview (first 10 shares ints) ----
    [49, 220, 165, 22, 228, 251, 229, 246, 47, 127]
  ---- no-fault reconstructed coeffs preview (first 10) ----
    [10, 148, 8, 220, 0, 0, 0, 0, 0, 0]
  ---- faulty reconstructed coeffs preview (first 10) ----
    [253, 57, 175, 3, 185, 252, 168, 69, 86, 38]
  ---- note ----
    Some assertions in the fault-randomization part are probabilistic;
    rare collisions can make them fail even if the construction is correct.
  AssertionError: [fault] coeff(5) not randomized (rare-collision possible): got=252 e

Running tests:  68%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉                                                                     | 68/100 [00:23<00:10,  3.02it/s]


[ASSERT FAIL] test_generate_A_hat_shamir
  ---- parameters ----
    n=10, indegree=7, outdegree=3, num_faults=2
  ---- support points (n_supports) preview ----
    n_supports (first 10): [23, 185, 122, 77, 192, 187, 239, 64, 49, 87]
  ---- secret+coeffs preview ----
    secrets_plus_indegree_poly_coefficients (as ints, first 10): [200, 103, 3, 107, 83, 156, 105, 82]
  ---- faults ----
    faulty_share_index_list: [4, 5]
  ---- shares_n_indegree preview (first 10 shares ints) ----
    [101, 124, 162, 223, 87, 5, 141, 149, 82, 234]
  ---- no-fault reconstructed coeffs preview (first 10) ----
    [200, 103, 3, 107, 0, 0, 0, 0, 0, 0]
  ---- faulty reconstructed coeffs preview (first 10) ----
    [236, 0, 40, 213, 251, 48, 146, 124, 85, 15]
  ---- note ----
    Some assertions in the fault-randomization part are probabilistic;
    rare collisions can make them fail even if the construction is correct.
  AssertionError: [fault] coeff(1) stayed zero (rare-collision possible): got=0. faulty_i

Running tests:  85%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                | 85/100 [00:28<00:04,  3.03it/s]


[ASSERT FAIL] test_generate_A_hat_shamir
  ---- parameters ----
    n=10, indegree=7, outdegree=3, num_faults=2
  ---- support points (n_supports) preview ----
    n_supports (first 10): [64, 205, 53, 48, 30, 12, 139, 50, 84, 11]
  ---- secret+coeffs preview ----
    secrets_plus_indegree_poly_coefficients (as ints, first 10): [185, 141, 197, 250, 155, 104, 184, 130]
  ---- faults ----
    faulty_share_index_list: [0, 7]
  ---- shares_n_indegree preview (first 10 shares ints) ----
    [78, 114, 246, 234, 13, 17, 11, 101, 140, 29]
  ---- no-fault reconstructed coeffs preview (first 10) ----
    [185, 141, 197, 250, 0, 0, 0, 0, 0, 0]
  ---- faulty reconstructed coeffs preview (first 10) ----
    [187, 106, 237, 15, 155, 93, 158, 64, 3, 142]
  ---- note ----
    Some assertions in the fault-randomization part are probabilistic;
    rare collisions can make them fail even if the construction is correct.
  AssertionError: [fault] coeff(4) not randomized (rare-collision possible): got=155 e

Running tests: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:33<00:00,  2.98it/s]


===== Summary =====
Total runs: 100
Pass:  97
Fail:  3
Error: 0

First few failures/errors (index, seed, message):
   (1, 2, "AssertionError('[fault] coeff(5) not randomized (rare-collision possible): got=252 equals original=252. faulty_indices=[1, 4]')")
   (67, 68, "AssertionError('[fault] coeff(1) stayed zero (rare-collision possible): got=0. faulty_indices=[4, 5]')")
   (84, 85, "AssertionError('[fault] coeff(4) not randomized (rare-collision possible): got=155 equals original=155. faulty_indices=[0, 7]')")
